<a href="https://colab.research.google.com/github/renatofb98/Data_science_projects/blob/main/Comparativo%20LOA%2027%20X%20dota%C3%A7%C3%A3o%2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import requests
import pandas as pd

# Município e demonstrativo
IBGE = 3511508
ANO = 2026
BIMESTRE = 4

URL = (
    "https://apidatalake.tesouro.gov.br"
    "/ords/cdwhprd/siconfi/tt/rreo"
)

parametros = {
    "an_exercicio": ANO,
    "nr_periodo": BIMESTRE,
    "co_tipo_demonstrativo": "RREO",
    "no_anexo": "RREO-Anexo 02",
    "id_ente": IBGE
}

# Consultar o SICONFI
resposta = requests.get(
    URL,
    params=parametros,
    timeout=60
)

resposta.raise_for_status()
dados = resposta.json()

# Dados da primeira página retornada
df = pd.DataFrame(dados.get("items", []))

print("Status HTTP:", resposta.status_code)
print("Registros nesta página:", len(df))
print("Existem outras páginas?:", dados.get("hasMore"))
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nPrimeiras 10 linhas:")
display(df.head(10))


Status HTTP: 200
Registros nesta página: 846
Existem outras páginas?: False

Colunas disponíveis:
['exercicio', 'demonstrativo', 'periodo', 'periodicidade', 'instituicao', 'cod_ibge', 'uf', 'populacao', 'anexo', 'esfera', 'rotulo', 'coluna', 'cod_conta', 'conta', 'valor']

Primeiras 10 linhas:


,exercicio,demonstrativo,periodo,periodicidade,instituicao,cod_ibge,uf,populacao,anexo,esfera,rotulo,coluna,cod_conta,conta,valor
0,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,DESPESAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I),291353650.0
1,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Legislativa,4085000.0
2,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Ação Legislativa,4085000.0
3,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Administração,9320000.0
4,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,FU04 - Administração Geral,9320000.0
5,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Segurança Pública,12552000.0
6,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Policiamento,11430000.0
7,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Defesa Civil,410000.0
8,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,FU06 - Administração Geral,712000.0
9,2026,RREO,4,B,Prefeitura Municipal de Cerquilho - SP,3511508,SP,46391,RREO-Anexo 02,M,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO INICIAL,RREO2TotalDespesas,Assistência Social,6419000.0


In [ ]:

# ETAPA 2 — CONHECER A ESTRUTURA DO RREO

print("COLUNAS DE VALORES DISPONÍVEIS:\n")

for coluna in df["coluna"].unique():
    print("-", coluna)

print("\n" + "=" * 65)

print("\nGRUPOS DE CONTAS DISPONÍVEIS:\n")

grupos = df[
    ["rotulo", "cod_conta"]
].drop_duplicates()

display(grupos)


COLUNAS DE VALORES DISPONÍVEIS:

- DOTAÇÃO INICIAL
- DOTAÇÃO ATUALIZADA (a)
- DESPESAS EMPENHADAS NO BIMESTRE
- DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)
- % (b/total b)
- SALDO (c) = (a-b)
- DESPESAS LIQUIDADAS NO BIMESTRE
- DESPESAS LIQUIDADAS ATÉ O BIMESTRE (d)
- % (d/total d)
- SALDO (e) = (a-d)
- % (b/III b)
- % (d/III d)


GRUPOS DE CONTAS DISPONÍVEIS:



,rotulo,cod_conta
0,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas
476,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra


In [ ]:

# ETAPA 3 — INSPECIONAR AS DESPESAS POR FUNÇÃO
# Não faz somas nem altera o DataFrame original.

# Funções presentes na proposta da LOA 2027
funcoes = [
    "Legislativa",
    "Administração",
    "Segurança Pública",
    "Assistência Social",
    "Previdência Social",
    "Saúde",
    "Trabalho",
    "Educação",
    "Cultura",
    "Urbanismo",
    "Saneamento",
    "Gestão Ambiental",
    "Desporto e Lazer",
    "Encargos Especiais",
    "Reserva de Contingência"
]

# Selecionar apenas as despesas empenhadas acumuladas
filtro_coluna = (
    df["coluna"] ==
    "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
)

# Selecionar apenas os nomes das funções
filtro_funcao = df["conta"].isin(funcoes)

# Preservar ambos os grupos de despesa
resultado = df.loc[
    filtro_coluna & filtro_funcao,
    ["rotulo", "cod_conta", "conta", "valor"]
].copy()

# Ordenar apenas para facilitar a leitura
resultado = resultado.sort_values(
    ["conta", "cod_conta"]
)

print("Registros encontrados:", len(resultado))
display(resultado)


Registros encontrados: 27


,rotulo,cod_conta,conta,valor
147,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Administração,8464463.22
590,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra,Administração,211198.52
153,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Assistência Social,4870551.87
595,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra,Assistência Social,116672.58
176,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Cultura,516225.50
614,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra,Cultura,22878.12
185,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Desporto e Lazer,3582412.19
622,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra,Desporto e Lazer,57773.88
168,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Educação,59059631.09
608,Total das Despesas Intra-Orçamentárias,RREO2TotalDespesasIntra,Educação,3357284.80


In [ ]:

# ETAPA 4 — CONFERÊNCIA DOS TOTAIS
# Utiliza df e resultado das etapas anteriores.
# Não modifica os dados originais.

coluna = "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"

# Somar as funções que identificamos
totais_funcoes = resultado.groupby(
    ["cod_conta", "rotulo"]
)["valor"].sum()

print("1. SOMA DAS FUNÇÕES IDENTIFICADAS")
display(totais_funcoes)

print("\n" + "=" * 60)

# Identificar os totais oficiais existentes no demonstrativo
totais_oficiais = df.loc[
    (df["coluna"] == coluna)
    & df["conta"].str.contains(
        r"^DESPESAS\s*\(",
        case=False,
        regex=True,
        na=False
    ),
    ["cod_conta", "rotulo", "conta", "valor"]
].copy()

print("2. TOTAIS INFORMADOS PELO SICONFI")
display(totais_oficiais)

print("\n" + "=" * 60)

# Conferir se os grupos coincidem
print("3. DIFERENÇAS ENTRE TOTAIS E SOMA DAS FUNÇÕES")

for codigo in [
    "RREO2TotalDespesas",
    "RREO2TotalDespesasIntra"
]:
    soma_funcoes = resultado.loc[
        resultado["cod_conta"] == codigo,
        "valor"
    ].sum()

    candidatos = totais_oficiais.loc[
        totais_oficiais["cod_conta"] == codigo
    ]

    if len(candidatos) == 1:
        total_oficial = candidatos.iloc[0]["valor"]
        diferenca = total_oficial - soma_funcoes

        print(f"\n{codigo}")
        print(f"Total oficial: R$ {total_oficial:,.2f}")
        print(f"Soma funções:  R$ {soma_funcoes:,.2f}")
        print(f"Diferença:     R$ {diferenca:,.2f}")
    else:
        print(
            f"\n{codigo}: encontrados "
            f"{len(candidatos)} totais candidatos. "
            "Necessário inspecionar."
        )


1. SOMA DAS FUNÇÕES IDENTIFICADAS


,,valor
cod_conta,rotulo,
RREO2TotalDespesas,Total das Despesas Exceto Intra-Orçamentárias,2.181186e+08
RREO2TotalDespesasIntra,Total das Despesas Intra-Orçamentárias,1.608165e+07



2. TOTAIS INFORMADOS PELO SICONFI


,cod_conta,rotulo,conta,valor
144,RREO2TotalDespesas,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I),2.181186e+08
189,RREO2TotalDespesas,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS (INTRA-ORÇAMENTÁRIAS) (II),1.608165e+07
587,RREO2TotalDespesasIntra,Total das Despesas Intra-Orçamentárias,DESPESAS (INTRA-ORÇAMENTÁRIAS) (II),1.608165e+07



3. DIFERENÇAS ENTRE TOTAIS E SOMA DAS FUNÇÕES

RREO2TotalDespesas: encontrados 2 totais candidatos. Necessário inspecionar.

RREO2TotalDespesasIntra
Total oficial: R$ 16,081,651.65
Soma funções:  R$ 16,081,651.65
Diferença:     R$ 0.00


In [ ]:

# ETAPA 4.1 — CONFERÊNCIA CORRIGIDA
# Não altera os dados originais.

coluna = "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"

contas_totais = {
    "RREO2TotalDespesas":
        "DESPESAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)",
    "RREO2TotalDespesasIntra":
        "DESPESAS (INTRA-ORÇAMENTÁRIAS) (II)"
}

total_geral = 0

for codigo, nome_conta in contas_totais.items():

    oficiais = df.loc[
        (df["coluna"] == coluna)
        & (df["cod_conta"] == codigo)
        & (df["conta"] == nome_conta),
        "valor"
    ]

    funcoes = resultado.loc[
        resultado["cod_conta"] == codigo,
        "valor"
    ]

    if len(oficiais) != 1:
        print(
            f"ERRO: {codigo} possui "
            f"{len(oficiais)} registros oficiais."
        )
        continue

    valor_oficial = float(oficiais.iloc[0])
    soma_funcoes = float(funcoes.sum())
    diferenca = valor_oficial - soma_funcoes

    print(f"\n{nome_conta}")
    print(f"Total oficial: R$ {valor_oficial:,.2f}")
    print(f"Soma funções:  R$ {soma_funcoes:,.2f}")
    print(f"Diferença:     R$ {diferenca:,.2f}")

    if abs(diferenca) < 0.01:
        print("CONFERÊNCIA: OK")
        total_geral += valor_oficial
    else:
        print("CONFERÊNCIA: REVISAR")

print("\n" + "=" * 55)
print(f"TOTAL GERAL CONFERIDO: R$ {total_geral:,.2f}")



DESPESAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)
Total oficial: R$ 218,118,570.61
Soma funções:  R$ 218,118,570.61
Diferença:     R$ 0.00
CONFERÊNCIA: OK

DESPESAS (INTRA-ORÇAMENTÁRIAS) (II)
Total oficial: R$ 16,081,651.65
Soma funções:  R$ 16,081,651.65
Diferença:     R$ 0.00
CONFERÊNCIA: OK

TOTAL GERAL CONFERIDO: R$ 234,200,222.26


In [ ]:


# ETAPA 5 — CORREÇÃO DA TABELA POR FUNÇÃO

# Recuperar a lista correta de funções
nomes_funcoes = [
    "Legislativa",
    "Administração",
    "Segurança Pública",
    "Assistência Social",
    "Previdência Social",
    "Saúde",
    "Trabalho",
    "Educação",
    "Cultura",
    "Urbanismo",
    "Saneamento",
    "Gestão Ambiental",
    "Desporto e Lazer",
    "Encargos Especiais",
    "Reserva de Contingência"
]

colunas = [
    "DOTAÇÃO INICIAL",
    "DOTAÇÃO ATUALIZADA (a)",
    "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
]

# Selecionar os registros correspondentes às funções
base = df.loc[
    df["conta"].isin(nomes_funcoes)
    & df["coluna"].isin(colunas)
    & df["cod_conta"].isin([
        "RREO2TotalDespesas",
        "RREO2TotalDespesasIntra"
    ]),
    ["conta", "coluna", "cod_conta", "valor"]
].copy()

# Conferir duplicidades antes de consolidar
chaves = ["conta", "coluna", "cod_conta"]

if base.duplicated(chaves).any():
    display(base[base.duplicated(chaves, keep=False)])
    raise ValueError("Existem registros duplicados.")

# Construir a tabela
tabela = base.pivot_table(
    index="conta",
    columns="coluna",
    values="valor",
    aggfunc="sum"
)

tabela = tabela.reindex(
    index=nomes_funcoes,
    columns=colunas
)

tabela.columns = [
    "Dotação inicial 2026",
    "Dotação atualizada ago/2026",
    "Empenhado até ago/2026"
]

tabela.index.name = "Função"
tabela = tabela.reset_index()

# Mostrar com duas casas decimais
display(
    tabela.style.format({
        "Dotação inicial 2026": "R$ {:,.2f}",
        "Dotação atualizada ago/2026": "R$ {:,.2f}",
        "Empenhado até ago/2026": "R$ {:,.2f}"
    }, na_rep="—")
)

print("\nTOTAIS:")
totais = tabela.iloc[:, 1:].sum()
display(totais)

# Validação do total empenhado
total_esperado = 234200222.26
total_encontrado = totais["Empenhado até ago/2026"]

if abs(total_encontrado - total_esperado) < 0.01:
    print("\nCONFERÊNCIA: OK")
else:
    print("\nATENÇÃO: o total empenhado não corresponde ao RREO.")


,Função,Dotação inicial 2026,Dotação atualizada ago/2026,Empenhado até ago/2026
0,Legislativa,"R$ 4,455,000.00","R$ 4,455,000.00","R$ 2,376,641.11"
1,Administração,"R$ 9,600,000.00","R$ 10,431,300.00","R$ 8,675,661.74"
2,Segurança Pública,"R$ 13,394,000.00","R$ 13,394,000.00","R$ 7,385,984.33"
3,Assistência Social,"R$ 6,603,000.00","R$ 7,366,586.89","R$ 4,987,224.45"
4,Previdência Social,"R$ 57,190,000.00","R$ 57,190,000.00","R$ 36,211,856.97"
5,Saúde,"R$ 69,250,000.00","R$ 75,163,576.53","R$ 62,174,380.03"
6,Trabalho,"R$ 3,721,000.00","R$ 3,721,000.00","R$ 2,337,371.75"
7,Educação,"R$ 91,418,000.00","R$ 94,869,225.95","R$ 62,416,915.89"
8,Cultura,"R$ 1,400,000.00","R$ 1,400,000.00","R$ 539,103.62"
9,Urbanismo,"R$ 15,576,000.00","R$ 23,181,558.40","R$ 20,828,584.75"



TOTAIS:


,0
Dotação inicial 2026,"R$ 315,000,000.00"
Dotação atualizada ago/2026,"R$ 333,744,443.00"
Empenhado até ago/2026,"R$ 234,200,222.26"



CONFERÊNCIA: OK


In [ ]:

# ETAPA 6 — COMPARAÇÃO LOA 2026 x PROPOSTA LOA 2027
# Não modifica a tabela original.

loa_2027 = {
    "Legislativa": 4885000,
    "Administração": 10508000,
    "Segurança Pública": 13016000,
    "Assistência Social": 7461000,
    "Previdência Social": 71972000,
    "Saúde": 77844000,
    "Trabalho": 3980000,
    "Educação": 94995000,
    "Cultura": 1415000,
    "Urbanismo": 16150000,
    "Saneamento": 24440000,
    "Gestão Ambiental": 9273000,
    "Desporto e Lazer": 4601000,
    "Encargos Especiais": 6500000,
    "Reserva de Contingência": 6500000
}

# Conferir se os valores totalizam a proposta oficial
assert sum(loa_2027.values()) == 353540000

# Criar uma cópia da tabela anterior
comparativo = tabela.copy()

# Adicionar a proposta 2027
comparativo["Proposta LOA 2027"] = (
    comparativo["Função"].map(loa_2027)
)

# Variação nominal contra o orçamento inicial de 2026
comparativo["Diferença R$"] = (
    comparativo["Proposta LOA 2027"]
    - comparativo["Dotação inicial 2026"]
)

# Variação percentual contra o orçamento inicial
comparativo["Variação %"] = (
    comparativo["Proposta LOA 2027"]
    / comparativo["Dotação inicial 2026"]
    - 1
) * 100

# Exibir somente as colunas desta comparação
colunas_exibir = [
    "Função",
    "Dotação inicial 2026",
    "Proposta LOA 2027",
    "Diferença R$",
    "Variação %"
]

display(
    comparativo[colunas_exibir].style.format({
        "Dotação inicial 2026": "R$ {:,.2f}",
        "Proposta LOA 2027": "R$ {:,.2f}",
        "Diferença R$": "R$ {:+,.2f}",
        "Variação %": "{:+.2f}%"
    })
)

print("\nCONFERÊNCIA:")
print(
    "Total LOA 2026:",
    comparativo["Dotação inicial 2026"].sum()
)
print(
    "Total proposta 2027:",
    comparativo["Proposta LOA 2027"].sum()
)


,Função,Dotação inicial 2026,Proposta LOA 2027,Diferença R$,Variação %
0,Legislativa,"R$ 4,455,000.00","R$ 4,885,000.00","R$ +430,000.00",+9.65%
1,Administração,"R$ 9,600,000.00","R$ 10,508,000.00","R$ +908,000.00",+9.46%
2,Segurança Pública,"R$ 13,394,000.00","R$ 13,016,000.00","R$ -378,000.00",-2.82%
3,Assistência Social,"R$ 6,603,000.00","R$ 7,461,000.00","R$ +858,000.00",+12.99%
4,Previdência Social,"R$ 57,190,000.00","R$ 71,972,000.00","R$ +14,782,000.00",+25.85%
5,Saúde,"R$ 69,250,000.00","R$ 77,844,000.00","R$ +8,594,000.00",+12.41%
6,Trabalho,"R$ 3,721,000.00","R$ 3,980,000.00","R$ +259,000.00",+6.96%
7,Educação,"R$ 91,418,000.00","R$ 94,995,000.00","R$ +3,577,000.00",+3.91%
8,Cultura,"R$ 1,400,000.00","R$ 1,415,000.00","R$ +15,000.00",+1.07%
9,Urbanismo,"R$ 15,576,000.00","R$ 16,150,000.00","R$ +574,000.00",+3.69%



CONFERÊNCIA:
Total LOA 2026: 315000000.0
Total proposta 2027: 353540000


In [ ]:

# ETAPA 7 — LOA 2027 x DOTAÇÃO ATUALIZADA DE 2026
# Apenas cria novas colunas na tabela comparativa.

comparativo["Diferença vs atualizada R$"] = (
    comparativo["Proposta LOA 2027"]
    - comparativo["Dotação atualizada ago/2026"]
)

comparativo["Variação vs atualizada %"] = (
    comparativo["Proposta LOA 2027"]
    / comparativo["Dotação atualizada ago/2026"]
    - 1
) * 100

colunas_exibir = [
    "Função",
    "Dotação atualizada ago/2026",
    "Proposta LOA 2027",
    "Diferença vs atualizada R$",
    "Variação vs atualizada %"
]

display(
    comparativo[colunas_exibir].style.format({
        "Dotação atualizada ago/2026": "R$ {:,.2f}",
        "Proposta LOA 2027": "R$ {:,.2f}",
        "Diferença vs atualizada R$": "R$ {:+,.2f}",
        "Variação vs atualizada %": "{:+.2f}%"
    })
)

print("\nCONFERÊNCIA DOS TOTAIS")

total_2026 = comparativo[
    "Dotação atualizada ago/2026"
].sum()

total_2027 = comparativo[
    "Proposta LOA 2027"
].sum()

diferenca = total_2027 - total_2026

print(f"Dotação atualizada 2026: R$ {total_2026:,.2f}")
print(f"Proposta LOA 2027:     R$ {total_2027:,.2f}")
print(f"Diferença:             R$ {diferenca:,.2f}")
print(f"Variação:              {diferenca/total_2026*100:.2f}%")


,Função,Dotação atualizada ago/2026,Proposta LOA 2027,Diferença vs atualizada R$,Variação vs atualizada %
0,Legislativa,"R$ 4,455,000.00","R$ 4,885,000.00","R$ +430,000.00",+9.65%
1,Administração,"R$ 10,431,300.00","R$ 10,508,000.00","R$ +76,700.00",+0.74%
2,Segurança Pública,"R$ 13,394,000.00","R$ 13,016,000.00","R$ -378,000.00",-2.82%
3,Assistência Social,"R$ 7,366,586.89","R$ 7,461,000.00","R$ +94,413.11",+1.28%
4,Previdência Social,"R$ 57,190,000.00","R$ 71,972,000.00","R$ +14,782,000.00",+25.85%
5,Saúde,"R$ 75,163,576.53","R$ 77,844,000.00","R$ +2,680,423.47",+3.57%
6,Trabalho,"R$ 3,721,000.00","R$ 3,980,000.00","R$ +259,000.00",+6.96%
7,Educação,"R$ 94,869,225.95","R$ 94,995,000.00","R$ +125,774.05",+0.13%
8,Cultura,"R$ 1,400,000.00","R$ 1,415,000.00","R$ +15,000.00",+1.07%
9,Urbanismo,"R$ 23,181,558.40","R$ 16,150,000.00","R$ -7,031,558.40",-30.33%



CONFERÊNCIA DOS TOTAIS
Dotação atualizada 2026: R$ 333,744,443.00
Proposta LOA 2027:     R$ 353,540,000.00
Diferença:             R$ 19,795,557.00
Variação:              5.93%


In [ ]:

# ETAPA 8 — EXTRAÇÃO DA EXECUÇÃO DE 2025
# RREO Anexo 02 — 4º e 6º bimestres

import requests
import pandas as pd

url = (
    "https://apidatalake.tesouro.gov.br"
    "/ords/cdwhprd/siconfi/tt/rreo"
)

nomes_funcoes = [
    "Legislativa",
    "Administração",
    "Segurança Pública",
    "Assistência Social",
    "Previdência Social",
    "Saúde",
    "Trabalho",
    "Educação",
    "Cultura",
    "Urbanismo",
    "Saneamento",
    "Gestão Ambiental",
    "Desporto e Lazer",
    "Encargos Especiais",
    "Reserva de Contingência"
]

coluna_empenho = (
    "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
)

def consultar_2025(bimestre):
    params = {
        "an_exercicio": 2025,
        "nr_periodo": bimestre,
        "co_tipo_demonstrativo": "RREO",
        "no_anexo": "RREO-Anexo 02",
        "id_ente": 3511508
    }

    resposta = requests.get(
        url, params=params, timeout=60
    )
    resposta.raise_for_status()

    dados = resposta.json()

    if dados.get("hasMore", False):
        raise RuntimeError(
            "Existem páginas adicionais. "
            "Precisamos paginar antes de continuar."
        )

    return pd.DataFrame(dados.get("items", []))


df_2025_b4 = consultar_2025(4)
df_2025_b6 = consultar_2025(6)

print("Registros B4:", len(df_2025_b4))
print("Registros B6:", len(df_2025_b6))


def extrair_empenhos(dados, nome):

    filtro = (
        dados["conta"].isin(nomes_funcoes)
        & (dados["coluna"] == coluna_empenho)
        & dados["cod_conta"].isin([
            "RREO2TotalDespesas",
            "RREO2TotalDespesasIntra"
        ])
    )

    selecionados = dados.loc[
        filtro,
        ["conta", "cod_conta", "valor"]
    ].copy()

    # Não somar registros duplicados silenciosamente
    if selecionados.duplicated(
        ["conta", "cod_conta"]
    ).any():
        raise ValueError(
            f"Duplicidades encontradas em {nome}"
        )

    totais = (
        selecionados.groupby("conta")["valor"]
        .sum()
        .reindex(nomes_funcoes)
    )

    return totais


empenhado_2025_b4 = extrair_empenhos(
    df_2025_b4, "2025-B4"
)

empenhado_2025_b6 = extrair_empenhos(
    df_2025_b6, "2025-B6"
)

conferencia_2025 = pd.DataFrame({
    "Função": nomes_funcoes,
    "Empenhado até ago/2025": empenhado_2025_b4.values,
    "Empenhado até dez/2025": empenhado_2025_b6.values
})

display(
    conferencia_2025.style.format({
        "Empenhado até ago/2025": "R$ {:,.2f}",
        "Empenhado até dez/2025": "R$ {:,.2f}"
    }, na_rep="—")
)

print("\nTOTAIS POR PERÍODO:")
print(
    conferencia_2025[
        ["Empenhado até ago/2025",
         "Empenhado até dez/2025"]
    ].sum()
)


Registros B4: 728
Registros B6: 723


,Função,Empenhado até ago/2025,Empenhado até dez/2025
0,Legislativa,"R$ 2,351,076.60","R$ 3,535,944.23"
1,Administração,"R$ 9,106,199.02","R$ 11,452,437.95"
2,Segurança Pública,"R$ 7,310,136.17","R$ 10,937,683.21"
3,Assistência Social,"R$ 9,872,809.77","R$ 11,496,467.39"
4,Previdência Social,"R$ 30,611,926.48","R$ 49,162,281.15"
5,Saúde,"R$ 62,843,711.38","R$ 73,222,411.41"
6,Trabalho,"R$ 2,432,148.08","R$ 3,621,023.52"
7,Educação,"R$ 59,192,137.86","R$ 92,556,684.02"
8,Cultura,"R$ 1,017,029.91","R$ 1,616,226.65"
9,Urbanismo,"R$ 11,226,215.03","R$ 13,697,288.82"



TOTAIS POR PERÍODO:
Empenhado até ago/2025   R$ 217,766,925.64
Empenhado até dez/2025   R$ 304,474,264.08
dtype: float64


In [ ]:

# BACKUP DA ANÁLISE ORÇAMENTÁRIA — CERQUILHO

import os
import pickle
from google.colab import drive

# Conectar ao Google Drive
drive.mount("/content/drive")

# Criar pasta para os backups
pasta = "/content/drive/MyDrive/Analise_Orcamentaria_Cerquilho"
os.makedirs(pasta, exist_ok=True)

# Salvar apenas as variáveis da análise
variaveis = [
    "df",
    "resultado",
    "tabela",
    "comparativo",
    "df_2025_b4",
    "df_2025_b6",
    "conferencia_2025",
    "empenhado_2025_b4",
    "empenhado_2025_b6",
    "nomes_funcoes",
    "loa_2027"
]

backup = {}

for nome in variaveis:
    if nome in globals():
        backup[nome] = globals()[nome]

arquivo = os.path.join(
    pasta, "backup_analise.pkl"
)

with open(arquivo, "wb") as f:
    pickle.dump(backup, f)

print("Backup realizado com sucesso!")
print("Arquivo:", arquivo)
print("Variáveis salvas:", list(backup.keys()))


Mounted at /content/drive
Backup realizado com sucesso!
Arquivo: /content/drive/MyDrive/Analise_Orcamentaria_Cerquilho/backup_analise.pkl
Variáveis salvas: ['df', 'resultado', 'tabela', 'comparativo', 'df_2025_b4', 'df_2025_b6', 'conferencia_2025', 'empenhado_2025_b4', 'empenhado_2025_b6', 'nomes_funcoes', 'loa_2027']


In [1]:

# RESTAURAR A ANÁLISE

import pickle
from google.colab import drive

drive.mount("/content/drive")

arquivo = (
    "/content/drive/MyDrive/"
    "Analise_Orcamentaria_Cerquilho/"
    "backup_analise.pkl"
)

with open(arquivo, "rb") as f:
    backup = pickle.load(f)

globals().update(backup)

print("Análise restaurada!")
print("Variáveis disponíveis:", list(backup.keys()))

display(conferencia_2025)
display(comparativo.head())


Mounted at /content/drive
Análise restaurada!
Variáveis disponíveis: ['df', 'resultado', 'tabela', 'comparativo', 'df_2025_b4', 'df_2025_b6', 'conferencia_2025', 'empenhado_2025_b4', 'empenhado_2025_b6', 'nomes_funcoes', 'loa_2027']


,Função,Empenhado até ago/2025,Empenhado até dez/2025
0,Legislativa,2351076.60,3535944.23
1,Administração,9106199.02,11452437.95
2,Segurança Pública,7310136.17,10937683.21
3,Assistência Social,9872809.77,11496467.39
4,Previdência Social,30611926.48,49162281.15
5,Saúde,62843711.38,73222411.41
6,Trabalho,2432148.08,3621023.52
7,Educação,59192137.86,92556684.02
8,Cultura,1017029.91,1616226.65
9,Urbanismo,11226215.03,13697288.82


,Função,Dotação inicial 2026,Dotação atualizada ago/2026,Empenhado até ago/2026,Proposta LOA 2027,Diferença R$,Variação %,Diferença vs atualizada R$,Variação vs atualizada %
0,Legislativa,4455000.0,4455000.00,2376641.11,4885000,430000.0,9.652076,430000.00,9.652076
1,Administração,9600000.0,10431300.00,8675661.74,10508000,908000.0,9.458333,76700.00,0.735287
2,Segurança Pública,13394000.0,13394000.00,7385984.33,13016000,-378000.0,-2.822159,-378000.00,-2.822159
3,Assistência Social,6603000.0,7366586.89,4987224.45,7461000,858000.0,12.994094,94413.11,1.281640
4,Previdência Social,57190000.0,57190000.00,36211856.97,71972000,14782000.0,25.847176,14782000.00,25.847176


In [2]:

# ETAPA 9 — PROJEÇÃO SAZONAL DO FECHAMENTO DE 2026
# Preserva as tabelas anteriores.

import pandas as pd
import numpy as np

projecao_2026 = comparativo.copy()

# Incorporar os empenhos de 2025 por nome da função
projecao_2026 = projecao_2026.merge(
    conferencia_2025,
    on="Função",
    how="left",
    validate="one_to_one"
)

# Proporção empenhada até agosto de 2025
projecao_2026["Proporção agosto/2025"] = (
    projecao_2026["Empenhado até ago/2025"]
    / projecao_2026["Empenhado até dez/2025"]
)

# Identificar funções sem base adequada
valida = (
    projecao_2026["Proporção agosto/2025"].notna()
    & np.isfinite(projecao_2026["Proporção agosto/2025"])
    & (projecao_2026["Proporção agosto/2025"] > 0)
)

# Projetar apenas funções com dados válidos
projecao_2026["Fechamento estimado 2026"] = np.nan

projecao_2026.loc[
    valida, "Fechamento estimado 2026"
] = (
    projecao_2026.loc[
        valida, "Empenhado até ago/2026"
    ]
    / projecao_2026.loc[
        valida, "Proporção agosto/2025"
    ]
)

# Diferença entre a projeção e dotação atualizada
projecao_2026["Diferença vs dotação"] = (
    projecao_2026["Fechamento estimado 2026"]
    - projecao_2026["Dotação atualizada ago/2026"]
)

# Alerta para valores acima da autorização
projecao_2026["Situação"] = np.select(
    [
        ~valida,
        projecao_2026["Diferença vs dotação"] > 0
    ],
    [
        "SEM BASE HISTÓRICA",
        "ACIMA DA DOTAÇÃO ATUALIZADA"
    ],
    default="DENTRO DA DOTAÇÃO ATUALIZADA"
)

colunas = [
    "Função",
    "Empenhado até ago/2026",
    "Proporção agosto/2025",
    "Fechamento estimado 2026",
    "Dotação atualizada ago/2026",
    "Diferença vs dotação",
    "Situação"
]

display(
    projecao_2026[colunas].style.format({
        "Empenhado até ago/2026": "R$ {:,.2f}",
        "Proporção agosto/2025": "{:.2%}",
        "Fechamento estimado 2026": "R$ {:,.2f}",
        "Dotação atualizada ago/2026": "R$ {:,.2f}",
        "Diferença vs dotação": "R$ {:+,.2f}"
    }, na_rep="—")
)

print("\nTOTAL DAS FUNÇÕES COM PROJEÇÃO:")
print(
    f"R$ {projecao_2026['Fechamento estimado 2026'].sum():,.2f}"
)

print("\nFUNÇÕES SEM PROJEÇÃO:")
print(
    projecao_2026.loc[
        ~valida, "Função"
    ].tolist()
)

print("\nATENÇÃO: o total acima é parcial se houver "
      "funções sem projeção.")


,Função,Empenhado até ago/2026,Proporção agosto/2025,Fechamento estimado 2026,Dotação atualizada ago/2026,Diferença vs dotação,Situação
0,Legislativa,"R$ 2,376,641.11",66.49%,"R$ 3,574,392.44","R$ 4,455,000.00","R$ -880,607.56",DENTRO DA DOTAÇÃO ATUALIZADA
1,Administração,"R$ 8,675,661.74",79.51%,"R$ 10,910,971.47","R$ 10,431,300.00","R$ +479,671.47",ACIMA DA DOTAÇÃO ATUALIZADA
2,Segurança Pública,"R$ 7,385,984.33",66.83%,"R$ 11,051,169.90","R$ 13,394,000.00","R$ -2,342,830.10",DENTRO DA DOTAÇÃO ATUALIZADA
3,Assistência Social,"R$ 4,987,224.45",85.88%,"R$ 5,807,410.92","R$ 7,366,586.89","R$ -1,559,175.97",DENTRO DA DOTAÇÃO ATUALIZADA
4,Previdência Social,"R$ 36,211,856.97",62.27%,"R$ 58,155,683.03","R$ 57,190,000.00","R$ +965,683.03",ACIMA DA DOTAÇÃO ATUALIZADA
5,Saúde,"R$ 62,174,380.03",85.83%,"R$ 72,442,539.34","R$ 75,163,576.53","R$ -2,721,037.19",DENTRO DA DOTAÇÃO ATUALIZADA
6,Trabalho,"R$ 2,337,371.75",67.17%,"R$ 3,479,918.90","R$ 3,721,000.00","R$ -241,081.10",DENTRO DA DOTAÇÃO ATUALIZADA
7,Educação,"R$ 62,416,915.89",63.95%,"R$ 97,599,157.09","R$ 94,869,225.95","R$ +2,729,931.14",ACIMA DA DOTAÇÃO ATUALIZADA
8,Cultura,"R$ 539,103.62",62.93%,"R$ 856,723.71","R$ 1,400,000.00","R$ -543,276.29",DENTRO DA DOTAÇÃO ATUALIZADA
9,Urbanismo,"R$ 20,828,584.75",81.96%,"R$ 25,413,297.38","R$ 23,181,558.40","R$ +2,231,738.98",ACIMA DA DOTAÇÃO ATUALIZADA



TOTAL DAS FUNÇÕES COM PROJEÇÃO:
R$ 322,915,446.78

FUNÇÕES SEM PROJEÇÃO:
['Encargos Especiais', 'Reserva de Contingência']

ATENÇÃO: o total acima é parcial se houver funções sem projeção.


In [3]:

# ETAPA 10 — INVESTIGAR ENCARGOS ESPECIAIS EM 2025
# Não altera nenhuma variável anterior.

for periodo, dados in [
    ("Agosto/2025", df_2025_b4),
    ("Dezembro/2025", df_2025_b6)
]:

    print("\n" + "=" * 55)
    print(periodo)

    filtro = dados["conta"].astype(str).str.contains(
        "ENCARGOS",
        case=False,
        na=False
    )

    encontrados = dados.loc[
        filtro,
        ["rotulo", "cod_conta", "conta", "coluna", "valor"]
    ].copy()

    print("Registros encontrados:", len(encontrados))

    if encontrados.empty:
        print("Nenhum registro com 'ENCARGOS' encontrado.")
    else:
        display(encontrados)



Agosto/2025
Registros encontrados: 0
Nenhum registro com 'ENCARGOS' encontrado.

Dezembro/2025
Registros encontrados: 0
Nenhum registro com 'ENCARGOS' encontrado.


In [4]:

# ETAPA 11 — INVESTIGAR ENCARGOS ESPECIAIS EM 2026

coluna = "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"

# Localizar a posição da função Encargos Especiais
dados_2026 = df.loc[
    (df["coluna"] == coluna)
    & (df["cod_conta"].isin([
        "RREO2TotalDespesas",
        "RREO2TotalDespesasIntra"
    ])),
    ["rotulo", "cod_conta", "conta", "valor"]
].copy()

# Exibir os registros de encargos e as subfunções
# relacionadas, conforme a ordem do demonstrativo

for rotulo, grupo in dados_2026.groupby("rotulo", sort=False):

    grupo = grupo.reset_index(drop=True)

    posicoes = grupo.index[
        grupo["conta"].eq("Encargos Especiais")
    ].tolist()

    if not posicoes:
        continue

    print("\n" + "=" * 55)
    print(rotulo)

    for pos in posicoes:
        print("\nFUNÇÃO E REGISTROS SEGUINTES:")
        display(
            grupo.iloc[pos:pos+8]
        )



Total das Despesas Exceto Intra-Orçamentárias

FUNÇÃO E REGISTROS SEGUINTES:


,rotulo,cod_conta,conta,valor
43,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Encargos Especiais,4.607939e+06
44,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,Outros Encargos Especiais,4.607939e+06
45,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,DESPESAS (INTRA-ORÇAMENTÁRIAS) (II),1.608165e+07
46,Total das Despesas Exceto Intra-Orçamentárias,RREO2TotalDespesas,TOTAL (III) = (I + II),2.342002e+08


In [5]:

# ETAPA 12 — TABELA CONSOLIDADA COM AJUSTES MANUAIS

import numpy as np
import pandas as pd

analise_final = projecao_2026.copy()

# Coluna para receber previsões oficiais/manualizadas
analise_final["Previsão informada 2026"] = np.nan

# Preservar o método de cada referência
analise_final["Método adotado"] = np.where(
    analise_final["Fechamento estimado 2026"].notna(),
    "Projeção sazonal 2025",
    "Sem estimativa"
)

# Referência inicial: projeção sazonal
analise_final["Referência adotada 2026"] = (
    analise_final["Fechamento estimado 2026"]
)

# Encargos: usar dotação como referência provisória
mask_encargos = (
    analise_final["Função"] == "Encargos Especiais"
)

analise_final.loc[
    mask_encargos, "Referência adotada 2026"
] = analise_final.loc[
    mask_encargos, "Dotação atualizada ago/2026"
]

analise_final.loc[
    mask_encargos, "Método adotado"
] = "Dotação atualizada (provisória)"

# Reserva: não projetar empenho
mask_reserva = (
    analise_final["Função"] == "Reserva de Contingência"
)

analise_final.loc[
    mask_reserva, "Método adotado"
] = "Não projetado"

# Diferença para a proposta de 2027
analise_final["Diferença 2027 vs referência"] = (
    analise_final["Proposta LOA 2027"]
    - analise_final["Referência adotada 2026"]
)

# Visualização
colunas = [
    "Função",
    "Fechamento estimado 2026",
    "Previsão informada 2026",
    "Referência adotada 2026",
    "Método adotado",
    "Proposta LOA 2027",
    "Diferença 2027 vs referência"
]

display(
    analise_final[colunas].style.format({
        "Fechamento estimado 2026": "R$ {:,.2f}",
        "Previsão informada 2026": "R$ {:,.2f}",
        "Referência adotada 2026": "R$ {:,.2f}",
        "Proposta LOA 2027": "R$ {:,.2f}",
        "Diferença 2027 vs referência": "R$ {:+,.2f}"
    }, na_rep="—")
)

print("\nTOTAL DAS REFERÊNCIAS ADOTADAS:")
print(
    f"R$ {analise_final['Referência adotada 2026'].sum():,.2f}"
)

print("\nOBSERVAÇÃO:")
print("O total mistura projeções sazonais e uma referência")
print("orçamentária provisória para Encargos Especiais.")
print("A Reserva de Contingência não está incluída.")


,Função,Fechamento estimado 2026,Previsão informada 2026,Referência adotada 2026,Método adotado,Proposta LOA 2027,Diferença 2027 vs referência
0,Legislativa,"R$ 3,574,392.44",—,"R$ 3,574,392.44",Projeção sazonal 2025,"R$ 4,885,000.00","R$ +1,310,607.56"
1,Administração,"R$ 10,910,971.47",—,"R$ 10,910,971.47",Projeção sazonal 2025,"R$ 10,508,000.00","R$ -402,971.47"
2,Segurança Pública,"R$ 11,051,169.90",—,"R$ 11,051,169.90",Projeção sazonal 2025,"R$ 13,016,000.00","R$ +1,964,830.10"
3,Assistência Social,"R$ 5,807,410.92",—,"R$ 5,807,410.92",Projeção sazonal 2025,"R$ 7,461,000.00","R$ +1,653,589.08"
4,Previdência Social,"R$ 58,155,683.03",—,"R$ 58,155,683.03",Projeção sazonal 2025,"R$ 71,972,000.00","R$ +13,816,316.97"
5,Saúde,"R$ 72,442,539.34",—,"R$ 72,442,539.34",Projeção sazonal 2025,"R$ 77,844,000.00","R$ +5,401,460.66"
6,Trabalho,"R$ 3,479,918.90",—,"R$ 3,479,918.90",Projeção sazonal 2025,"R$ 3,980,000.00","R$ +500,081.10"
7,Educação,"R$ 97,599,157.09",—,"R$ 97,599,157.09",Projeção sazonal 2025,"R$ 94,995,000.00","R$ -2,604,157.09"
8,Cultura,"R$ 856,723.71",—,"R$ 856,723.71",Projeção sazonal 2025,"R$ 1,415,000.00","R$ +558,276.29"
9,Urbanismo,"R$ 25,413,297.38",—,"R$ 25,413,297.38",Projeção sazonal 2025,"R$ 16,150,000.00","R$ -9,263,297.38"



TOTAL DAS REFERÊNCIAS ADOTADAS:
R$ 328,480,446.78

OBSERVAÇÃO:
O total mistura projeções sazonais e uma referência
orçamentária provisória para Encargos Especiais.
A Reserva de Contingência não está incluída.


In [6]:

# ETAPA 13 — AJUSTES MANUAIS DA PROJEÇÃO 2026
# Não altera a tabela original analise_final.

# Digite aqui os valores informados pela Secretaria.
# Deixe o dicionário vazio enquanto não houver informações.

previsoes_manuais = {
    # "Educação": 95000000,
    # "Saúde": 74000000,
}

analise_ajustada = analise_final.copy()

# Conferir se os nomes digitados existem
nomes_validos = set(analise_ajustada["Função"])

desconhecidos = (
    set(previsoes_manuais) - nomes_validos
)

if desconhecidos:
    raise ValueError(
        f"Funções não encontradas: {desconhecidos}"
    )

# Registrar previsões informadas
analise_ajustada["Previsão informada 2026"] = (
    analise_ajustada["Função"].map(previsoes_manuais)
)

# Substituir apenas os valores informados
tem_previsao = (
    analise_ajustada["Previsão informada 2026"].notna()
)

analise_ajustada.loc[
    tem_previsao, "Referência adotada 2026"
] = analise_ajustada.loc[
    tem_previsao, "Previsão informada 2026"
]

analise_ajustada.loc[
    tem_previsao, "Método adotado"
] = "Previsão informada"

# Recalcular a diferença para 2027
analise_ajustada["Diferença 2027 vs referência"] = (
    analise_ajustada["Proposta LOA 2027"]
    - analise_ajustada["Referência adotada 2026"]
)

# Calcular a variação percentual
analise_ajustada["Variação 2027 vs referência %"] = (
    analise_ajustada["Diferença 2027 vs referência"]
    / analise_ajustada["Referência adotada 2026"]
) * 100

colunas = [
    "Função",
    "Fechamento estimado 2026",
    "Previsão informada 2026",
    "Referência adotada 2026",
    "Método adotado",
    "Proposta LOA 2027",
    "Variação 2027 vs referência %"
]

display(
    analise_ajustada[colunas].style.format({
        "Fechamento estimado 2026": "R$ {:,.2f}",
        "Previsão informada 2026": "R$ {:,.2f}",
        "Referência adotada 2026": "R$ {:,.2f}",
        "Proposta LOA 2027": "R$ {:,.2f}",
        "Variação 2027 vs referência %": "{:+.2f}%"
    }, na_rep="—")
)

print("\nPrevisões manuais inseridas:", len(previsoes_manuais))
print("Tabela original preservada.")


,Função,Fechamento estimado 2026,Previsão informada 2026,Referência adotada 2026,Método adotado,Proposta LOA 2027,Variação 2027 vs referência %
0,Legislativa,"R$ 3,574,392.44",—,"R$ 3,574,392.44",Projeção sazonal 2025,"R$ 4,885,000.00",+36.67%
1,Administração,"R$ 10,910,971.47",—,"R$ 10,910,971.47",Projeção sazonal 2025,"R$ 10,508,000.00",-3.69%
2,Segurança Pública,"R$ 11,051,169.90",—,"R$ 11,051,169.90",Projeção sazonal 2025,"R$ 13,016,000.00",+17.78%
3,Assistência Social,"R$ 5,807,410.92",—,"R$ 5,807,410.92",Projeção sazonal 2025,"R$ 7,461,000.00",+28.47%
4,Previdência Social,"R$ 58,155,683.03",—,"R$ 58,155,683.03",Projeção sazonal 2025,"R$ 71,972,000.00",+23.76%
5,Saúde,"R$ 72,442,539.34",—,"R$ 72,442,539.34",Projeção sazonal 2025,"R$ 77,844,000.00",+7.46%
6,Trabalho,"R$ 3,479,918.90",—,"R$ 3,479,918.90",Projeção sazonal 2025,"R$ 3,980,000.00",+14.37%
7,Educação,"R$ 97,599,157.09",—,"R$ 97,599,157.09",Projeção sazonal 2025,"R$ 94,995,000.00",-2.67%
8,Cultura,"R$ 856,723.71",—,"R$ 856,723.71",Projeção sazonal 2025,"R$ 1,415,000.00",+65.16%
9,Urbanismo,"R$ 25,413,297.38",—,"R$ 25,413,297.38",Projeção sazonal 2025,"R$ 16,150,000.00",-36.45%



Previsões manuais inseridas: 0
Tabela original preservada.


In [7]:

# ETAPA 14 — DETALHAMENTO DE URBANISMO EM 2026
# Apenas consulta o df já restaurado.

colunas_urbanismo = [
    "DOTAÇÃO ATUALIZADA (a)",
    "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
]

dados_urbanismo = df.loc[
    df["coluna"].isin(colunas_urbanismo)
    & df["cod_conta"].isin([
        "RREO2TotalDespesas",
        "RREO2TotalDespesasIntra"
    ]),
    ["rotulo", "coluna", "conta", "valor"]
].copy()

for rotulo in [
    "Total das Despesas Exceto Intra-Orçamentárias",
    "Total das Despesas Intra-Orçamentárias"
]:
    for coluna in colunas_urbanismo:

        grupo = dados_urbanismo.loc[
            (dados_urbanismo["rotulo"] == rotulo)
            & (dados_urbanismo["coluna"] == coluna)
        ].reset_index(drop=True)

        posicoes = grupo.index[
            grupo["conta"] == "Urbanismo"
        ].tolist()

        if not posicoes:
            continue

        print("\n" + "=" * 60)
        print(rotulo)
        print(coluna)

        inicio = posicoes[0]

        # Mostrar a função e os registros seguintes
        display(
            grupo.iloc[inicio:inicio + 8]
        )



Total das Despesas Exceto Intra-Orçamentárias
DOTAÇÃO ATUALIZADA (a)


,rotulo,coluna,conta,valor
35,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Urbanismo,22781558.40
36,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Infra-Estrutura Urbana,8055958.40
37,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Serviços Urbanos,14725600.00
38,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Saneamento,20488150.00
39,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Saneamento Básico Urbano,20488150.00
40,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Gestão Ambiental,7427695.23
41,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Preservação e Conservação Ambiental,7427695.23
42,Total das Despesas Exceto Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Desporto e Lazer,5789500.00



Total das Despesas Exceto Intra-Orçamentárias
DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)


,rotulo,coluna,conta,valor
34,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Urbanismo,20543558.53
35,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Infra-Estrutura Urbana,7770887.36
36,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Serviços Urbanos,12772671.17
37,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Saneamento,11753343.16
38,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Saneamento Básico Urbano,11753343.16
39,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Gestão Ambiental,4834354.56
40,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Preservação e Conservação Ambiental,4834354.56
41,Total das Despesas Exceto Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Desporto e Lazer,3582412.19



Total das Despesas Intra-Orçamentárias
DOTAÇÃO ATUALIZADA (a)


,rotulo,coluna,conta,valor
29,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Urbanismo,400000.0
30,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Serviços Urbanos,400000.0
31,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Saneamento,1961850.0
32,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Saneamento Básico Urbano,1961850.0
33,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Gestão Ambiental,250000.0
34,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Preservação e Conservação Ambiental,250000.0
35,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Desporto e Lazer,90000.0
36,Total das Despesas Intra-Orçamentárias,DOTAÇÃO ATUALIZADA (a),Desporto Comunitário,90000.0



Total das Despesas Intra-Orçamentárias
DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)


,rotulo,coluna,conta,valor
29,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Urbanismo,285026.22
30,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Serviços Urbanos,285026.22
31,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Saneamento,1252712.15
32,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Saneamento Básico Urbano,1252712.15
33,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Gestão Ambiental,177962.70
34,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Preservação e Conservação Ambiental,177962.70
35,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Desporto e Lazer,57773.88
36,Total das Despesas Intra-Orçamentárias,DESPESAS EMPENHADAS ATÉ O BIMESTRE (b),Desporto Comunitário,57773.88


In [8]:

# ETAPA 15 — ALTERAÇÕES ORÇAMENTÁRIAS DE URBANISMO
# Não modifica as tabelas anteriores.

subfuncoes = [
    "Infra-Estrutura Urbana",
    "Serviços Urbanos"
]

colunas = [
    "DOTAÇÃO INICIAL",
    "DOTAÇÃO ATUALIZADA (a)"
]

filtro = (
    df["conta"].isin(subfuncoes)
    & df["coluna"].isin(colunas)
    & df["cod_conta"].isin([
        "RREO2TotalDespesas",
        "RREO2TotalDespesasIntra"
    ])
)

base_urbanismo = df.loc[
    filtro,
    ["conta", "coluna", "cod_conta", "valor"]
].copy()

# Conferir duplicidades
if base_urbanismo.duplicated(
    ["conta", "coluna", "cod_conta"]
).any():
    raise ValueError("Registros duplicados encontrados.")

# Consolidar despesas intra e não intra
alteracoes_urbanismo = base_urbanismo.pivot_table(
    index="conta",
    columns="coluna",
    values="valor",
    aggfunc="sum"
).reindex(index=subfuncoes, columns=colunas)

alteracoes_urbanismo.columns = [
    "Dotação inicial",
    "Dotação atualizada"
]

alteracoes_urbanismo["Alteração líquida"] = (
    alteracoes_urbanismo["Dotação atualizada"]
    - alteracoes_urbanismo["Dotação inicial"]
)

alteracoes_urbanismo["Variação %"] = (
    alteracoes_urbanismo["Alteração líquida"]
    / alteracoes_urbanismo["Dotação inicial"]
) * 100

display(
    alteracoes_urbanismo.style.format({
        "Dotação inicial": "R$ {:,.2f}",
        "Dotação atualizada": "R$ {:,.2f}",
        "Alteração líquida": "R$ {:+,.2f}",
        "Variação %": "{:+.2f}%"
    }, na_rep="—")
)

print("\nCONFERÊNCIA:")
print(
    "Alteração líquida total: R$ "
    f"{alteracoes_urbanismo['Alteração líquida'].sum():,.2f}"
)

print(
    "Valor esperado: R$ 7,605,558.40"
)


,Dotação inicial,Dotação atualizada,Alteração líquida,Variação %
conta,,,,
Infra-Estrutura Urbana,"R$ 2,265,000.00","R$ 8,055,958.40","R$ +5,790,958.40",+255.67%
Serviços Urbanos,"R$ 13,311,000.00","R$ 15,125,600.00","R$ +1,814,600.00",+13.63%



CONFERÊNCIA:
Alteração líquida total: R$ 7,605,558.40
Valor esperado: R$ 7,605,558.40


In [9]:

# ETAPA 16 — ATUALIZAÇÃO DOS DADOS OFICIAIS
# Não altera a análise existente.

import requests
import pandas as pd
import os

ANO = 2026
BIMESTRE_ALVO = 5  # Alterar para 6 no fechamento anual

url = (
    "https://apidatalake.tesouro.gov.br"
    "/ords/cdwhprd/siconfi/tt/rreo"
)

params = {
    "an_exercicio": ANO,
    "nr_periodo": BIMESTRE_ALVO,
    "co_tipo_demonstrativo": "RREO",
    "no_anexo": "RREO-Anexo 02",
    "id_ente": 3511508
}

resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()

dados_api = resposta.json()

if dados_api.get("hasMore", False):
    raise RuntimeError("Existem páginas adicionais na API.")

df_novo = pd.DataFrame(dados_api.get("items", []))

# Verificar se os registros solicitados foram encontrados
if df_novo.empty:
    print("O demonstrativo solicitado ainda não foi encontrado.")

else:
    periodos = pd.to_numeric(
        df_novo["periodo"], errors="coerce"
    )

    exercicios = pd.to_numeric(
        df_novo["exercicio"], errors="coerce"
    )

    if not (
        periodos.eq(BIMESTRE_ALVO).all()
        and exercicios.eq(ANO).all()
    ):
        raise ValueError("Ano ou bimestre diferente do solicitado.")

    print("Demonstrativo encontrado!")
    print("Ano:", ANO)
    print("Bimestre:", BIMESTRE_ALVO)
    print("Registros:", len(df_novo))

    # Salvar cópia independente no Drive
    pasta = (
        "/content/drive/MyDrive/"
        "Analise_Orcamentaria_Cerquilho"
    )

    os.makedirs(pasta, exist_ok=True)

    arquivo = os.path.join(
        pasta,
        f"RREO_Anexo02_{ANO}_B{BIMESTRE_ALVO}.csv"
    )

    df_novo.to_csv(
        arquivo,
        index=False,
        encoding="utf-8-sig"
    )

    print("Dados salvos em:", arquivo)
    print("Análise anterior preservada.")


O demonstrativo solicitado ainda não foi encontrado.


In [10]:

# ETAPA 17 — COMPARATIVO CONSOLIDADO 2026 x 2027
# Preserva todas as tabelas anteriores.

consolidado = analise_ajustada.copy()

consolidado["Variação orçamento %"] = (
    consolidado["Proposta LOA 2027"]
    / consolidado["Dotação atualizada ago/2026"]
    - 1
) * 100

consolidado["Variação projeção %"] = (
    consolidado["Proposta LOA 2027"]
    / consolidado["Referência adotada 2026"]
    - 1
) * 100

colunas = [
    "Função",
    "Dotação atualizada ago/2026",
    "Referência adotada 2026",
    "Método adotado",
    "Proposta LOA 2027",
    "Variação orçamento %",
    "Variação projeção %"
]

display(
    consolidado[colunas].style.format({
        "Dotação atualizada ago/2026": "R$ {:,.2f}",
        "Referência adotada 2026": "R$ {:,.2f}",
        "Proposta LOA 2027": "R$ {:,.2f}",
        "Variação orçamento %": "{:+.2f}%",
        "Variação projeção %": "{:+.2f}%"
    }, na_rep="—")
)

print("\nCONFERÊNCIA:")
print(
    "Dotação atualizada 2026:",
    f"R$ {consolidado['Dotação atualizada ago/2026'].sum():,.2f}"
)
print(
    "Proposta LOA 2027:",
    f"R$ {consolidado['Proposta LOA 2027'].sum():,.2f}"
)
print(
    "Referências 2026 (sem reserva):",
    f"R$ {consolidado['Referência adotada 2026'].sum():,.2f}"
)


,Função,Dotação atualizada ago/2026,Referência adotada 2026,Método adotado,Proposta LOA 2027,Variação orçamento %,Variação projeção %
0,Legislativa,"R$ 4,455,000.00","R$ 3,574,392.44",Projeção sazonal 2025,"R$ 4,885,000.00",+9.65%,+36.67%
1,Administração,"R$ 10,431,300.00","R$ 10,910,971.47",Projeção sazonal 2025,"R$ 10,508,000.00",+0.74%,-3.69%
2,Segurança Pública,"R$ 13,394,000.00","R$ 11,051,169.90",Projeção sazonal 2025,"R$ 13,016,000.00",-2.82%,+17.78%
3,Assistência Social,"R$ 7,366,586.89","R$ 5,807,410.92",Projeção sazonal 2025,"R$ 7,461,000.00",+1.28%,+28.47%
4,Previdência Social,"R$ 57,190,000.00","R$ 58,155,683.03",Projeção sazonal 2025,"R$ 71,972,000.00",+25.85%,+23.76%
5,Saúde,"R$ 75,163,576.53","R$ 72,442,539.34",Projeção sazonal 2025,"R$ 77,844,000.00",+3.57%,+7.46%
6,Trabalho,"R$ 3,721,000.00","R$ 3,479,918.90",Projeção sazonal 2025,"R$ 3,980,000.00",+6.96%,+14.37%
7,Educação,"R$ 94,869,225.95","R$ 97,599,157.09",Projeção sazonal 2025,"R$ 94,995,000.00",+0.13%,-2.67%
8,Cultura,"R$ 1,400,000.00","R$ 856,723.71",Projeção sazonal 2025,"R$ 1,415,000.00",+1.07%,+65.16%
9,Urbanismo,"R$ 23,181,558.40","R$ 25,413,297.38",Projeção sazonal 2025,"R$ 16,150,000.00",-30.33%,-36.45%



CONFERÊNCIA:
Dotação atualizada 2026: R$ 333,744,443.00
Proposta LOA 2027: R$ 353,540,000.00
Referências 2026 (sem reserva): R$ 328,480,446.78


In [11]:

# ETAPA 18 — RANKING DAS ALTERAÇÕES ORÇAMENTÁRIAS
# Preserva o consolidado original.

ranking = consolidado.copy()

ranking["Diferença orçamento R$"] = (
    ranking["Proposta LOA 2027"]
    - ranking["Dotação atualizada ago/2026"]
)

ranking["Diferença projeção R$"] = (
    ranking["Proposta LOA 2027"]
    - ranking["Referência adotada 2026"]
)

# Ordenar pelo impacto absoluto em reais
ranking["Impacto absoluto"] = (
    ranking["Diferença orçamento R$"].abs()
)

ranking = ranking.sort_values(
    "Impacto absoluto",
    ascending=False
).reset_index(drop=True)

colunas = [
    "Função",
    "Diferença orçamento R$",
    "Variação orçamento %",
    "Diferença projeção R$",
    "Variação projeção %"
]

display(
    ranking[colunas].style.format({
        "Diferença orçamento R$": "R$ {:+,.2f}",
        "Variação orçamento %": "{:+.2f}%",
        "Diferença projeção R$": "R$ {:+,.2f}",
        "Variação projeção %": "{:+.2f}%"
    }, na_rep="—")
)

print("\nCONFERÊNCIA DA ALTERAÇÃO LÍQUIDA:")
print(
    f"R$ {ranking['Diferença orçamento R$'].sum():,.2f}"
)

print("Valor esperado: R$ 19,795,557.00")


,Função,Diferença orçamento R$,Variação orçamento %,Diferença projeção R$,Variação projeção %
0,Previdência Social,"R$ +14,782,000.00",+25.85%,"R$ +13,816,316.97",+23.76%
1,Urbanismo,"R$ -7,031,558.40",-30.33%,"R$ -9,263,297.38",-36.45%
2,Reserva de Contingência,"R$ +5,500,000.00",+550.00%,—,—
3,Saúde,"R$ +2,680,423.47",+3.57%,"R$ +5,401,460.66",+7.46%
4,Saneamento,"R$ +1,990,000.00",+8.86%,"R$ +2,054,249.80",+9.18%
5,Gestão Ambiental,"R$ +1,595,304.77",+20.78%,"R$ +2,264,233.08",+32.31%
6,Desporto e Lazer,"R$ -1,278,500.00",-21.75%,"R$ +371,334.53",+8.78%
7,Encargos Especiais,"R$ +935,000.00",+16.80%,"R$ +935,000.00",+16.80%
8,Legislativa,"R$ +430,000.00",+9.65%,"R$ +1,310,607.56",+36.67%
9,Segurança Pública,"R$ -378,000.00",-2.82%,"R$ +1,964,830.10",+17.78%



CONFERÊNCIA DA ALTERAÇÃO LÍQUIDA:
R$ 19,795,557.00
Valor esperado: R$ 19,795,557.00


In [13]:

# ETAPA 19 — EXCEL AUDITÁVEL
# Dados originais nas bases.
# Cálculos feitos por fórmulas do Excel.

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.workbook.properties import CalcProperties
from google.colab import files
import pandas as pd
import math

# 1. CONFERIR VARIÁVEIS DE ORIGEM

fontes = {
    "RREO_2025_B4": df_2025_b4,
    "RREO_2025_B6": df_2025_b6,
    "RREO_2026_B4": df
}

assert len(df_2025_b4) == 728
assert len(df_2025_b6) == 723
assert len(df) == 846
assert sum(loa_2027.values()) == 353540000

# 2. CRIAR EXCEL

wb = Workbook()
wb.remove(wb.active)

azul = "17365D"
azul_claro = "DCEAF7"
verde = "008060"

formato_rs = '"R$" #,##0.00;[Red]("R$" #,##0.00)'
formato_pct = '0.00%;[Red](0.00%)'

# 3. COPIAR OS DADOS BRUTOS DO SICONFI

for nome_aba, dados in fontes.items():

    ws = wb.create_sheet(nome_aba)

    # Copiar cabeçalhos originais
    ws.append(list(dados.columns))

    # Copiar registros sem consolidar ou calcular valores
    for registro in dados.itertuples(index=False, name=None):
        linha = []
        for valor in registro:
            if pd.isna(valor):
                linha.append(None)
            elif hasattr(valor, "item"):
                linha.append(valor.item())
            else:
                linha.append(valor)
        ws.append(linha)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for celula in ws[1]:
        celula.fill = PatternFill(
            "solid", fgColor=azul
        )
        celula.font = Font(
            color="FFFFFF", bold=True
        )

    for coluna in ws.columns:
        letra = coluna[0].column_letter
        ws.column_dimensions[letra].width = 20

    # Identificar as colunas pelos nomes originais
    mapa = {
        celula.value: celula.column
        for celula in ws[1]
    }

    ws.column_dimensions[
        get_column_letter(mapa["conta"])
    ].width = 43

    ws.column_dimensions[
        get_column_letter(mapa["coluna"])
    ].width = 43

    coluna_valor = mapa["valor"]

    for linha in range(2, ws.max_row + 1):
        ws.cell(
            linha, coluna_valor
        ).number_format = formato_rs


# 4. BASE DE ENTRADA DA LOA 2027

ws_loa = wb.create_sheet("LOA_2027")

ws_loa.append([
    "Função",
    "Proposta LOA 2027",
    "Fonte"
])

for funcao, valor in loa_2027.items():
    ws_loa.append([
        funcao,
        valor,
        "Projeto de Lei nº 11/2026"
    ])

ws_loa.freeze_panes = "A2"
ws_loa.auto_filter.ref = ws_loa.dimensions

ws_loa.column_dimensions["A"].width = 30
ws_loa.column_dimensions["B"].width = 24
ws_loa.column_dimensions["C"].width = 35

for linha in range(2, ws_loa.max_row + 1):
    ws_loa[f"B{linha}"].number_format = formato_rs
    ws_loa[f"B{linha}"].font = Font(color="0000FF")

for celula in ws_loa[1]:
    celula.fill = PatternFill("solid", fgColor=azul)
    celula.font = Font(color="FFFFFF", bold=True)


# 5. PLANILHA DE CÁLCULOS

ws = wb.create_sheet("CALCULOS")

cabecalhos = [
    "Função",
    "Empenhado ago/2025",
    "Empenhado dez/2025",
    "Proporção 2025",
    "Dotação inicial 2026",
    "Dotação atualizada 2026",
    "Empenhado ago/2026",
    "Projeção sazonal 2026",
    "Previsão manual 2026",
    "Referência adotada 2026",
    "Método",
    "Proposta LOA 2027",
    "Diferença vs dotação R$",
    "Variação vs dotação %",
    "Diferença vs referência R$",
    "Variação vs referência %"
]

ws.append(cabecalhos)

# As funções são identificadas a partir da proposta.
# Não utilizamos os resultados já calculados pelo Python.

funcoes_excel = list(loa_2027.keys())

# Identificar posições das colunas das bases originais

def referencias_base(nome):
    origem = wb[nome]

    mapa = {
        celula.value: celula.column_letter
        for celula in origem[1]
    }

    ultima = origem.max_row

    def intervalo(campo):
        letra = mapa[campo]
        return f"'{nome}'!${letra}$2:${letra}${ultima}"

    return {
        "conta": intervalo("conta"),
        "coluna": intervalo("coluna"),
        "codigo": intervalo("cod_conta"),
        "valor": intervalo("valor")
    }

base25_b4 = referencias_base("RREO_2025_B4")
base25_b6 = referencias_base("RREO_2025_B6")
base26 = referencias_base("RREO_2026_B4")


# Fórmula que soma apenas linhas de função:
# despesas intra + despesas exceto intra.
#
# Não soma subfunções, evitando dupla contagem.

def formula_siconfi(base, linha, campo):

    partes = []

    for codigo in [
        "RREO2TotalDespesas",
        "RREO2TotalDespesasIntra"
    ]:

        partes.append(
            f'SUMIFS({base["valor"]},'
            f'{base["conta"]},$A{linha},'
            f'{base["coluna"]},"{campo}",'
            f'{base["codigo"]},"{codigo}")'
        )

    soma = "+".join(partes)

    return (
        f'=IF(COUNTIFS({base["conta"]},$A{linha},'
        f'{base["coluna"]},"{campo}")=0,'
        f'"",{soma})'
    )


# 6. INSERIR FÓRMULAS

for indice, funcao in enumerate(funcoes_excel, start=2):

    ws[f"A{indice}"] = funcao

    # Dados de 2025 — obtidos via SUMIFS nas bases
    ws[f"B{indice}"] = formula_siconfi(
        base25_b4, indice,
        "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
    )

    ws[f"C{indice}"] = formula_siconfi(
        base25_b6, indice,
        "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
    )

    # Proporção sazonal de 2025
    ws[f"D{indice}"] = (
        f'=IFERROR(B{indice}/C{indice},"")'
    )

    # Dotação inicial 2026
    ws[f"E{indice}"] = formula_siconfi(
        base26, indice,
        "DOTAÇÃO INICIAL"
    )

    # Dotação atualizada 2026
    ws[f"F{indice}"] = formula_siconfi(
        base26, indice,
        "DOTAÇÃO ATUALIZADA (a)"
    )

    # Empenhos acumulados de 2026
    ws[f"G{indice}"] = formula_siconfi(
        base26, indice,
        "DESPESAS EMPENHADAS ATÉ O BIMESTRE (b)"
    )

    # Projeção sazonal 2026
    ws[f"H{indice}"] = (
        f'=IFERROR(G{indice}/D{indice},"")'
    )

    # Coluna I: previsão manual (fica vazia)

    # Referência adotada:
    # previsão manual > projeção sazonal > dotação
    # provisória de Encargos Especiais
    ws[f"J{indice}"] = (
        f'=IF(ISNUMBER(I{indice}),I{indice},'
        f'IF(ISNUMBER(H{indice}),H{indice},'
        f'IF(A{indice}="Encargos Especiais",'
        f'F{indice},"")))'
    )

    # Identificar método usado
    ws[f"K{indice}"] = (
        f'=IF(ISNUMBER(I{indice}),"Previsão manual",'
        f'IF(ISNUMBER(H{indice}),"Projeção sazonal",'
        f'IF(A{indice}="Encargos Especiais",'
        f'"Dotação provisória","Não projetado")))'
    )

    # LOA 2027: buscar na aba de entrada
    ws[f"L{indice}"] = (
        f'=IFERROR(INDEX(LOA_2027!$B$2:$B$16,'
        f'MATCH(A{indice},LOA_2027!$A$2:$A$16,0)),"")'
    )

    # Diferenças orçamentárias
    ws[f"M{indice}"] = f'=L{indice}-F{indice}'

    ws[f"N{indice}"] = (
        f'=IFERROR(L{indice}/F{indice}-1,"")'
    )

    # Diferenças frente à referência de fechamento
    ws[f"O{indice}"] = (
        f'=IF(ISNUMBER(J{indice}),'
        f'L{indice}-J{indice},"")'
    )

    ws[f"P{indice}"] = (
        f'=IFERROR(L{indice}/J{indice}-1,"")'
    )

# 7. TOTALIZADORES

linha_total = 17

ws[f"A{linha_total}"] = "TOTAL"

for coluna in [
    "B", "C", "E", "F", "G",
    "H", "J", "L", "M", "O"
]:
    ws[f"{coluna}{linha_total}"] = (
        f"=SUM({coluna}2:{coluna}16)"
    )

ws["N17"] = '=IFERROR(L17/F17-1,"")'

# Projeção mista exclui reserva; não tratar
# como fechamento homogêneo.
ws["P17"] = None

# 8. FORMATAÇÃO

ws.freeze_panes = "B2"
ws.auto_filter.ref = "A1:P16"

ws.column_dimensions["A"].width = 28
ws.column_dimensions["K"].width = 24

for coluna in "BCDEFGHIJLMNOP":
    ws.column_dimensions[coluna].width = 23

for celula in ws[1]:
    celula.fill = PatternFill("solid", fgColor=azul)
    celula.font = Font(color="FFFFFF", bold=True)
    celula.alignment = Alignment(
        wrap_text=True,
        vertical="center"
    )

ws.row_dimensions[1].height = 42

for linha in range(2, 18):

    for coluna in [
        "B", "C", "E", "F", "G",
        "H", "I", "J", "L", "M", "O"
    ]:
        ws[f"{coluna}{linha}"].number_format = formato_rs

    for coluna in ["D", "N", "P"]:
        ws[f"{coluna}{linha}"].number_format = formato_pct

    # Campo editável destacado
    ws[f"I{linha}"].fill = PatternFill(
        "solid", fgColor="FFF2CC"
    )

for celula in ws[linha_total]:
    celula.font = Font(bold=True)
    celula.fill = PatternFill(
        "solid", fgColor=azul_claro
    )

# 9. CONFERÊNCIAS QUE SERÃO CALCULADAS NO EXCEL

ws_check = wb.create_sheet("CONFERENCIA")

ws_check.append([
    "Indicador", "Calculado no Excel",
    "Valor de controle", "Diferença"
])

controles = [
    ("Dotação inicial 2026", "=CALCULOS!E17", 315000000),
    ("Dotação atualizada 2026", "=CALCULOS!F17", 333744443),
    ("Empenhado ago/2026", "=CALCULOS!G17", 234200222.26),
    ("Empenhado dez/2025", "=CALCULOS!C17", 304474264.08),
    ("Proposta LOA 2027", "=CALCULOS!L17", 353540000)
]

for linha, (nome, formula, controle) in enumerate(
    controles, start=2
):
    ws_check[f"A{linha}"] = nome
    ws_check[f"B{linha}"] = formula
    ws_check[f"C{linha}"] = controle
    ws_check[f"D{linha}"] = f"=B{linha}-C{linha}"

ws_check.column_dimensions["A"].width = 34

for coluna in "BCD":
    ws_check.column_dimensions[coluna].width = 25

for celula in ws_check[1]:
    celula.fill = PatternFill("solid", fgColor=azul)
    celula.font = Font(color="FFFFFF", bold=True)

for linha in range(2, 7):
    for coluna in "BCD":
        ws_check[f"{coluna}{linha}"].number_format = formato_rs

# 10. SOLICITAR RECÁLCULO AO ABRIR NO EXCEL

wb.calculation = CalcProperties(
    calcMode="auto",
    fullCalcOnLoad=True,
    forceFullCalc=True
)

# 11. SALVAR NO DRIVE

caminho = (
    "/content/drive/MyDrive/"
    "Analise_Orcamentaria_Cerquilho/"
    "Comparativo_2026_2027_Auditavel.xlsx"
)

wb.save(caminho)

print("PLANILHA GERADA")
print("Arquivo:", caminho)
print("Abas:", wb.sheetnames)

print("\nREGISTROS ORIGINAIS:")
for nome, dados in fontes.items():
    print(nome, len(dados))

print("\nATENÇÃO:")
print("Abra no Excel para recalcular as fórmulas.")
print("Confira a aba CONFERENCIA.")
print("Todas as diferenças devem ser R$ 0,00.")


PLANILHA GERADA
Arquivo: /content/drive/MyDrive/Analise_Orcamentaria_Cerquilho/Comparativo_2026_2027_Auditavel.xlsx
Abas: ['RREO_2025_B4', 'RREO_2025_B6', 'RREO_2026_B4', 'LOA_2027', 'CALCULOS', 'CONFERENCIA']

REGISTROS ORIGINAIS:
RREO_2025_B4 728
RREO_2025_B6 723
RREO_2026_B4 846

ATENÇÃO:
Abra no Excel para recalcular as fórmulas.
Confira a aba CONFERENCIA.
Todas as diferenças devem ser R$ 0,00.


In [14]:

from google.colab import files
import os

arquivo = (
    "/content/drive/MyDrive/"
    "Analise_Orcamentaria_Cerquilho/"
    "Comparativo_2026_2027_Auditavel.xlsx"
)

# Verificar se o arquivo realmente existe
if os.path.isfile(arquivo):
    print("Arquivo encontrado!")
    print(
        "Tamanho:",
        round(os.path.getsize(arquivo) / 1024, 2),
        "KB"
    )

    # Baixar para o computador
    files.download(arquivo)

else:
    print("ERRO: arquivo não encontrado no Google Drive.")


Arquivo encontrado!
Tamanho: 162.76 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:

# ANÁLISE POR SECRETARIA — ETAPA 1
# Testar a MSC Orçamentária de Cerquilho
# Não altera as variáveis anteriores.

import requests
import pandas as pd

url_msc = (
    "https://apidatalake.tesouro.gov.br"
    "/ords/cdwhprd/siconfi/tt/msc_orcamentaria"
)

parametros_msc = {
    "id_ente": 3511508,
    "an_referencia": 2026,
    "me_referencia": 8,
    "co_tipo_matriz": "MSCC",
    "classe_conta": 6,
    "id_tv": "ending_balance"
}

resposta_msc = requests.get(
    url_msc,
    params=parametros_msc,
    timeout=90
)

print("Status HTTP:", resposta_msc.status_code)
resposta_msc.raise_for_status()

conteudo_msc = resposta_msc.json()

df_msc_teste = pd.DataFrame(
    conteudo_msc.get("items", [])
)

print("Registros retornados:", len(df_msc_teste))
print("Há mais páginas?", conteudo_msc.get("hasMore"))

print("\nCOLUNAS DISPONÍVEIS:")
print(df_msc_teste.columns.tolist())

print("\nAMOSTRA DOS REGISTROS:")
display(df_msc_teste.head(5))


Status HTTP: 200
Registros retornados: 1832
Há mais páginas? False

COLUNAS DISPONÍVEIS:
['tipo_matriz', 'cod_ibge', 'classe_conta', 'conta_contabil', 'poder_orgao', 'ano_fonte_recursos', 'fonte_recursos', 'funcao', 'subfuncao', 'exercicio', 'mes_referencia', 'educacao_saude', 'data_referencia', 'entrada_msc', 'natureza_despesa', 'ano_inscricao', 'natureza_receita', 'valor', 'natureza_conta', 'tipo_valor', 'complemento_fonte']

AMOSTRA DOS REGISTROS:


,tipo_matriz,cod_ibge,classe_conta,conta_contabil,poder_orgao,ano_fonte_recursos,fonte_recursos,funcao,subfuncao,exercicio,...,educacao_saude,data_referencia,entrada_msc,natureza_despesa,ano_inscricao,natureza_receita,valor,natureza_conta,tipo_valor,complemento_fonte
0,MSCC,3511508,6,621100000,10131,1.0,1500,None,None,2026,...,None,2026-08-31T00:00:00Z,2620,None,NaN,16110301,97952.15,C,ending_balance,1002
1,MSCC,3511508,6,621100000,10131,1.0,1500,None,None,2026,...,None,2026-08-31T00:00:00Z,2623,None,NaN,16110302,865.04,C,ending_balance,1002
2,MSCC,3511508,6,621100000,10131,1.0,1500,None,None,2026,...,None,2026-08-31T00:00:00Z,2626,None,NaN,16110303,20623.29,C,ending_balance,1002
3,MSCC,3511508,6,621100000,10131,1.0,1500,None,None,2026,...,None,2026-08-31T00:00:00Z,2629,None,NaN,16110304,4463.36,C,ending_balance,1002
4,MSCC,3511508,6,621100000,10131,1.0,1500,None,None,2026,...,None,2026-08-31T00:00:00Z,2632,None,NaN,19999921,2085.00,D,ending_balance,1002


In [3]:

# ANÁLISE POR SECRETARIA — ETAPA 2
# Identificar os órgãos informados na MSC.
# Não altera os dados originais.

print("TOTAL DE REGISTROS:", len(df_msc_teste))

# Preservar os códigos como texto
codigos = (
    df_msc_teste["poder_orgao"]
    .astype("string")
    .str.strip()
)

resumo_orgaos = (
    codigos.value_counts(dropna=False)
    .rename_axis("Código poder_orgao")
    .reset_index(name="Quantidade de registros")
)

print("\nQUANTIDADE DE CÓDIGOS DIFERENTES:")
print(codigos.nunique(dropna=True))

print("\nCÓDIGOS ENCONTRADOS:")
display(resumo_orgaos)

print("\nREGISTROS SEM CÓDIGO:")
print(codigos.isna().sum())


TOTAL DE REGISTROS: 1832

QUANTIDADE DE CÓDIGOS DIFERENTES:
3

CÓDIGOS ENCONTRADOS:


,Código poder_orgao,Quantidade de registros
0,10131,1708
1,20231,90
2,10132,34



REGISTROS SEM CÓDIGO:
0


In [5]:

# ETAPA 4 — EXTRAÇÃO DOS TOTALIZADORES POR SECRETARIA
# Não altera nenhuma das bases anteriores.

!pip -q install pdfplumber

import pdfplumber
import pandas as pd
import re
from google.colab import files

# 1. Enviar o PDF do balancete de julho/2026
arquivos = files.upload()
caminho_pdf = next(iter(arquivos))

# 2. Identificar valores no formato brasileiro
padrao_valor = r"-?[\d.]+,\d{2}"

def converter_valor(texto):
    return float(
        texto.replace(".", "").replace(",", ".")
    )

# 3. Extrair apenas os totalizadores
registros = []
codigo_unidade = None

with pdfplumber.open(caminho_pdf) as pdf:

    for pagina, documento in enumerate(pdf.pages, start=1):

        texto = documento.extract_text() or ""

        for linha in texto.splitlines():

            # Identificar unidade orçamentária
            identificacao = re.match(
                r"^\s*(02\.\d{2})\s+(.+)",
                linha
            )

            if identificacao:
                codigo_unidade = identificacao.group(1)

            # Selecionar somente os totais da unidade
            if "Total por Unidade Orçamentária" in linha:

                trecho = linha.split(
                    "Total por Unidade Orçamentária", 1
                )[1]

                valores = re.findall(
                    padrao_valor, trecho
                )

                # O relatório possui 16 campos financeiros
                if len(valores) != 16:
                    raise ValueError(
                        f"Página {pagina}: encontrados "
                        f"{len(valores)} valores; esperados 16."
                    )

                if codigo_unidade is None:
                    raise ValueError(
                        f"Unidade não identificada na página {pagina}."
                    )

                registros.append({
                    "Código": codigo_unidade,
                    "Página PDF": pagina,
                    "Dotação inicial": converter_valor(valores[0]),
                    "Alteração líquida": converter_valor(valores[1]),
                    "Dotação atualizada": converter_valor(valores[2]),
                    "Empenhado até julho": converter_valor(valores[6])
                })

# 4. Criar tabela sem fazer projeções
secretarias_julho = pd.DataFrame(registros)

print("UNIDADES IDENTIFICADAS:", len(secretarias_julho))

display(
    secretarias_julho.style.format({
        "Dotação inicial": "R$ {:,.2f}",
        "Alteração líquida": "R$ {:+,.2f}",
        "Dotação atualizada": "R$ {:,.2f}",
        "Empenhado até julho": "R$ {:,.2f}"
    })
)

# 5. Conferir totais com o balancete
print("\nCONFERÊNCIA DOS TOTAIS")

for coluna in [
    "Dotação inicial",
    "Dotação atualizada",
    "Empenhado até julho"
]:
    print(
        f"{coluna}: "
        f"R$ {secretarias_julho[coluna].sum():,.2f}"
    )

print("\nVALORES DE CONTROLE DO BALANCETE:")
print("Dotação inicial: R$ 288,545,000.00")
print("Dotação atualizada: R$ 303,992,427.31")
print("Empenhado até julho: R$ 196,506,138.54")


Saving 394d9791d49484b888a207f2d5afb3c3.pdf to 394d9791d49484b888a207f2d5afb3c3.pdf
UNIDADES IDENTIFICADAS: 8


,Código,Página PDF,Dotação inicial,Alteração líquida,Dotação atualizada,Empenhado até julho
0,02.01,3,"R$ 73,355,000.00","R$ +745,300.00","R$ 74,100,300.00","R$ 44,098,013.49"
1,02.02,10,"R$ 92,818,000.00","R$ +3,451,225.95","R$ 96,269,225.95","R$ 56,388,587.77"
2,02.03,19,"R$ 75,853,000.00","R$ +5,510,147.73","R$ 81,363,147.73","R$ 63,791,488.23"
3,02.04,21,"R$ 15,576,000.00","R$ +5,095,253.63","R$ 20,671,253.63","R$ 15,886,505.24"
4,02.05,22,"R$ 8,388,000.00","R$ +150,000.00","R$ 8,538,000.00","R$ 4,507,974.63"
5,02.06,24,"R$ 13,394,000.00",R$ +0.00,"R$ 13,394,000.00","R$ 6,365,658.02"
6,02.07,25,"R$ 5,440,000.00","R$ +495,500.00","R$ 5,935,500.00","R$ 3,407,635.22"
7,02.08,26,"R$ 3,721,000.00",R$ +0.00,"R$ 3,721,000.00","R$ 2,060,275.94"



CONFERÊNCIA DOS TOTAIS
Dotação inicial: R$ 288,545,000.00
Dotação atualizada: R$ 303,992,427.31
Empenhado até julho: R$ 196,506,138.54

VALORES DE CONTROLE DO BALANCETE:
Dotação inicial: R$ 288,545,000.00
Dotação atualizada: R$ 303,992,427.31
Empenhado até julho: R$ 196,506,138.54


In [6]:

# ETAPA 5 — IDENTIFICAR AS SECRETARIAS
# Mantém a tabela original intacta.

nomes_secretarias = {
    "02.01": "Administração e Finanças",
    "02.02": "Educação e Cultura",
    "02.03": "Saúde e Promoção Social",
    "02.04": "Obras, Viação e Serviços",
    "02.05": "Agricultura e Meio Ambiente",
    "02.06": "Trânsito, Segurança e Defesa Civil",
    "02.07": "Esportes, Turismo, Lazer e Juventude",
    "02.08": "Trabalho e Desenvolvimento Econômico"
}

secretarias_identificadas = secretarias_julho.copy()

secretarias_identificadas.insert(
    1,
    "Secretaria",
    secretarias_identificadas["Código"].map(nomes_secretarias)
)

# Conferir se todos os códigos foram identificados
if secretarias_identificadas["Secretaria"].isna().any():
    raise ValueError("Existem códigos sem secretaria identificada.")

display(
    secretarias_identificadas.style.format({
        "Dotação inicial": "R$ {:,.2f}",
        "Alteração líquida": "R$ {:+,.2f}",
        "Dotação atualizada": "R$ {:,.2f}",
        "Empenhado até julho": "R$ {:,.2f}"
    })
)

print("\nTOTAL DE SECRETARIAS:",
      secretarias_identificadas["Secretaria"].nunique())


,Código,Secretaria,Página PDF,Dotação inicial,Alteração líquida,Dotação atualizada,Empenhado até julho
0,02.01,Administração e Finanças,3,"R$ 73,355,000.00","R$ +745,300.00","R$ 74,100,300.00","R$ 44,098,013.49"
1,02.02,Educação e Cultura,10,"R$ 92,818,000.00","R$ +3,451,225.95","R$ 96,269,225.95","R$ 56,388,587.77"
2,02.03,Saúde e Promoção Social,19,"R$ 75,853,000.00","R$ +5,510,147.73","R$ 81,363,147.73","R$ 63,791,488.23"
3,02.04,"Obras, Viação e Serviços",21,"R$ 15,576,000.00","R$ +5,095,253.63","R$ 20,671,253.63","R$ 15,886,505.24"
4,02.05,Agricultura e Meio Ambiente,22,"R$ 8,388,000.00","R$ +150,000.00","R$ 8,538,000.00","R$ 4,507,974.63"
5,02.06,"Trânsito, Segurança e Defesa Civil",24,"R$ 13,394,000.00",R$ +0.00,"R$ 13,394,000.00","R$ 6,365,658.02"
6,02.07,"Esportes, Turismo, Lazer e Juventude",25,"R$ 5,440,000.00","R$ +495,500.00","R$ 5,935,500.00","R$ 3,407,635.22"
7,02.08,Trabalho e Desenvolvimento Econômico,26,"R$ 3,721,000.00",R$ +0.00,"R$ 3,721,000.00","R$ 2,060,275.94"



TOTAL DE SECRETARIAS: 8


In [7]:

# ETAPA 6 — PROPOSTA LOA 2027 POR SECRETARIA
# Valores transcritos do projeto de lei.
# Conferir com o documento original.

import pandas as pd

loa_2027_secretarias = {
    "02.01": 95480000,
    "02.02": 96410000,
    "02.03": 85305000,
    "02.04": 16150000,
    "02.05": 9273000,
    "02.06": 13016000,
    "02.07": 4601000,
    "02.08": 3980000
}

# Criar tabela usando os nomes já validados
proposta_2027 = secretarias_identificadas[
    ["Código", "Secretaria"]
].copy()

proposta_2027["LOA proposta 2027"] = (
    proposta_2027["Código"].map(
        loa_2027_secretarias
    )
)

# Conferir se todas as unidades foram identificadas
assert proposta_2027["LOA proposta 2027"].notna().all()

display(
    proposta_2027.style.format({
        "LOA proposta 2027": "R$ {:,.2f}"
    })
)

print("\nTOTAL DAS SECRETARIAS DO EXECUTIVO:")
print(
    f"R$ {proposta_2027['LOA proposta 2027'].sum():,.2f}"
)

# Valores das outras entidades, mantidos separados
camara_2027 = 4885000
saaec_2027 = 24440000

total_geral = (
    proposta_2027["LOA proposta 2027"].sum()
    + camara_2027
    + saaec_2027
)

print("\nTOTAL GERAL DA PROPOSTA:")
print(f"R$ {total_geral:,.2f}")

assert total_geral == 353540000
print("CONFERÊNCIA: OK")


,Código,Secretaria,LOA proposta 2027
0,02.01,Administração e Finanças,"R$ 95,480,000.00"
1,02.02,Educação e Cultura,"R$ 96,410,000.00"
2,02.03,Saúde e Promoção Social,"R$ 85,305,000.00"
3,02.04,"Obras, Viação e Serviços","R$ 16,150,000.00"
4,02.05,Agricultura e Meio Ambiente,"R$ 9,273,000.00"
5,02.06,"Trânsito, Segurança e Defesa Civil","R$ 13,016,000.00"
6,02.07,"Esportes, Turismo, Lazer e Juventude","R$ 4,601,000.00"
7,02.08,Trabalho e Desenvolvimento Econômico,"R$ 3,980,000.00"



TOTAL DAS SECRETARIAS DO EXECUTIVO:
R$ 324,215,000.00

TOTAL GERAL DA PROPOSTA:
R$ 353,540,000.00
CONFERÊNCIA: OK


In [8]:

# ETAPA 7 — COMPARATIVO POR SECRETARIA
# Preserva as bases originais.

comparativo_secretarias = secretarias_identificadas.merge(
    proposta_2027[["Código", "LOA proposta 2027"]],
    on="Código",
    how="left",
    validate="one_to_one"
)

# Diferença nominal
comparativo_secretarias["Diferença R$"] = (
    comparativo_secretarias["LOA proposta 2027"]
    - comparativo_secretarias["Dotação atualizada"]
)

# Variação percentual
comparativo_secretarias["Variação %"] = (
    comparativo_secretarias["Diferença R$"]
    / comparativo_secretarias["Dotação atualizada"]
) * 100

# Participação no orçamento total proposto
comparativo_secretarias["Participação LOA 2027 %"] = (
    comparativo_secretarias["LOA proposta 2027"]
    / 353540000
) * 100

colunas = [
    "Secretaria",
    "Dotação atualizada",
    "LOA proposta 2027",
    "Diferença R$",
    "Variação %",
    "Participação LOA 2027 %"
]

display(
    comparativo_secretarias[colunas].style.format({
        "Dotação atualizada": "R$ {:,.2f}",
        "LOA proposta 2027": "R$ {:,.2f}",
        "Diferença R$": "R$ {:+,.2f}",
        "Variação %": "{:+.2f}%",
        "Participação LOA 2027 %": "{:.2f}%"
    })
)

print("\nCONFERÊNCIA — SOMENTE AS OITO SECRETARIAS")
print(
    "Dotação atualizada julho/2026:",
    f"R$ {comparativo_secretarias['Dotação atualizada'].sum():,.2f}"
)
print(
    "Proposta LOA 2027:",
    f"R$ {comparativo_secretarias['LOA proposta 2027'].sum():,.2f}"
)
print(
    "Diferença:",
    f"R$ {comparativo_secretarias['Diferença R$'].sum():,.2f}"
)


,Secretaria,Dotação atualizada,LOA proposta 2027,Diferença R$,Variação %,Participação LOA 2027 %
0,Administração e Finanças,"R$ 74,100,300.00","R$ 95,480,000.00","R$ +21,379,700.00",+28.85%,27.01%
1,Educação e Cultura,"R$ 96,269,225.95","R$ 96,410,000.00","R$ +140,774.05",+0.15%,27.27%
2,Saúde e Promoção Social,"R$ 81,363,147.73","R$ 85,305,000.00","R$ +3,941,852.27",+4.84%,24.13%
3,"Obras, Viação e Serviços","R$ 20,671,253.63","R$ 16,150,000.00","R$ -4,521,253.63",-21.87%,4.57%
4,Agricultura e Meio Ambiente,"R$ 8,538,000.00","R$ 9,273,000.00","R$ +735,000.00",+8.61%,2.62%
5,"Trânsito, Segurança e Defesa Civil","R$ 13,394,000.00","R$ 13,016,000.00","R$ -378,000.00",-2.82%,3.68%
6,"Esportes, Turismo, Lazer e Juventude","R$ 5,935,500.00","R$ 4,601,000.00","R$ -1,334,500.00",-22.48%,1.30%
7,Trabalho e Desenvolvimento Econômico,"R$ 3,721,000.00","R$ 3,980,000.00","R$ +259,000.00",+6.96%,1.13%



CONFERÊNCIA — SOMENTE AS OITO SECRETARIAS
Dotação atualizada julho/2026: R$ 303,992,427.31
Proposta LOA 2027: R$ 324,215,000.00
Diferença: R$ 20,222,572.69


In [9]:

# ETAPA 8 — ESTRUTURA MUNICIPAL COMPLETA
# Não altera a tabela comparativa original.

import pandas as pd

municipal = comparativo_secretarias.copy()

# Incluir entidades que não constam no
# balancete do Poder Executivo.

entidades = pd.DataFrame([
    {
        "Código": "01",
        "Secretaria": "Câmara Municipal",
        "Dotação inicial": 4455000,
        "Dotação atualizada": float("nan"),
        "LOA proposta 2027": 4885000
    },
    {
        "Código": "03",
        "Secretaria": "SAAEC",
        "Dotação inicial": 22000000,
        "Dotação atualizada": float("nan"),
        "LOA proposta 2027": 24440000
    }
])

municipal = pd.concat(
    [municipal, entidades],
    ignore_index=True
)

# Calcular somente onde há dados
municipal["Diferença R$"] = (
    municipal["LOA proposta 2027"]
    - municipal["Dotação atualizada"]
)

municipal["Variação %"] = (
    municipal["Diferença R$"]
    / municipal["Dotação atualizada"]
) * 100

display(
    municipal[
        [
            "Código",
            "Secretaria",
            "Dotação inicial",
            "Dotação atualizada",
            "LOA proposta 2027",
            "Diferença R$",
            "Variação %"
        ]
    ].style.format({
        "Dotação inicial": "R$ {:,.2f}",
        "Dotação atualizada": "R$ {:,.2f}",
        "LOA proposta 2027": "R$ {:,.2f}",
        "Diferença R$": "R$ {:+,.2f}",
        "Variação %": "{:+.2f}%"
    }, na_rep="PENDENTE")
)

print("\nCONFERÊNCIA:")
print(
    "LOA proposta 2027:",
    f"R$ {municipal['LOA proposta 2027'].sum():,.2f}"
)

print(
    "Entidades sem dotação atualizada:",
    municipal.loc[
        municipal["Dotação atualizada"].isna(),
        "Secretaria"
    ].tolist()
)


,Código,Secretaria,Dotação inicial,Dotação atualizada,LOA proposta 2027,Diferença R$,Variação %
0,02.01,Administração e Finanças,"R$ 73,355,000.00","R$ 74,100,300.00","R$ 95,480,000.00","R$ +21,379,700.00",+28.85%
1,02.02,Educação e Cultura,"R$ 92,818,000.00","R$ 96,269,225.95","R$ 96,410,000.00","R$ +140,774.05",+0.15%
2,02.03,Saúde e Promoção Social,"R$ 75,853,000.00","R$ 81,363,147.73","R$ 85,305,000.00","R$ +3,941,852.27",+4.84%
3,02.04,"Obras, Viação e Serviços","R$ 15,576,000.00","R$ 20,671,253.63","R$ 16,150,000.00","R$ -4,521,253.63",-21.87%
4,02.05,Agricultura e Meio Ambiente,"R$ 8,388,000.00","R$ 8,538,000.00","R$ 9,273,000.00","R$ +735,000.00",+8.61%
5,02.06,"Trânsito, Segurança e Defesa Civil","R$ 13,394,000.00","R$ 13,394,000.00","R$ 13,016,000.00","R$ -378,000.00",-2.82%
6,02.07,"Esportes, Turismo, Lazer e Juventude","R$ 5,440,000.00","R$ 5,935,500.00","R$ 4,601,000.00","R$ -1,334,500.00",-22.48%
7,02.08,Trabalho e Desenvolvimento Econômico,"R$ 3,721,000.00","R$ 3,721,000.00","R$ 3,980,000.00","R$ +259,000.00",+6.96%
8,01,Câmara Municipal,"R$ 4,455,000.00",PENDENTE,"R$ 4,885,000.00",PENDENTE,PENDENTE
9,03,SAAEC,"R$ 22,000,000.00",PENDENTE,"R$ 24,440,000.00",PENDENTE,PENDENTE



CONFERÊNCIA:
LOA proposta 2027: R$ 353,540,000.00
Entidades sem dotação atualizada: ['Câmara Municipal', 'SAAEC']


In [10]:

# ETAPA 9 — HISTÓRICO POR SECRETARIA (2025)
# Não altera nenhuma das tabelas anteriores.

import pdfplumber
import pandas as pd
import re
from google.colab import files

print("Envie os balancetes de JULHO e DEZEMBRO de 2025.")
arquivos_2025 = files.upload()

padrao_valor = r"-?[\d.]+,\d{2}"

def valor_br(texto):
    return float(texto.replace(".", "").replace(",", "."))

def extrair_unidades(caminho):
    registros = []
    codigo = None

    with pdfplumber.open(caminho) as pdf:
        cabecalho = (pdf.pages[0].extract_text() or "").upper()

        if "EXERCÍCIO: 2025" not in cabecalho:
            raise ValueError("O arquivo não é de 2025.")

        if "MÊS DE JULHO" in cabecalho:
            periodo = "Julho 2025"
        elif "MÊS DE DEZEMBRO" in cabecalho:
            periodo = "Dezembro 2025"
        else:
            raise ValueError("Período não identificado.")

        for pagina, folha in enumerate(pdf.pages, start=1):
            for linha in (folha.extract_text() or "").splitlines():

                identificacao = re.match(
                    r"^\s*(02\.\d{2})\s+(.+)", linha
                )
                if identificacao:
                    codigo = identificacao.group(1)

                if "Total por Unidade Orçamentária" in linha:
                    trecho = linha.split(
                        "Total por Unidade Orçamentária", 1
                    )[1]
                    valores = re.findall(padrao_valor, trecho)

                    if len(valores) != 16 or codigo is None:
                        raise ValueError(
                            f"Erro de leitura na página {pagina}"
                        )

                    registros.append({
                        "Código": codigo,
                        "Período": periodo,
                        "Página PDF": pagina,
                        "Empenhado acumulado": valor_br(valores[6])
                    })

    if len(registros) != 8:
        raise ValueError(
            f"{periodo}: esperadas 8 unidades, "
            f"encontradas {len(registros)}."
        )

    return pd.DataFrame(registros)

bases_2025 = [
    extrair_unidades(nome)
    for nome in arquivos_2025
]

historico_2025 = pd.concat(bases_2025, ignore_index=True)

# Conferência dos dois períodos
totais = historico_2025.groupby(
    "Período"
)["Empenhado acumulado"].sum()

controles = {
    "Julho 2025": 183545888.90,
    "Dezembro 2025": 280532572.59
}

for periodo, esperado in controles.items():
    obtido = totais.get(periodo, 0)
    print(f"{periodo}: R$ {obtido:,.2f}")

    assert round(obtido, 2) == round(esperado, 2), (
        f"Total divergente em {periodo}"
    )

# Organizar os valores lado a lado
empenhos_2025 = historico_2025.pivot(
    index="Código",
    columns="Período",
    values="Empenhado acumulado"
).reset_index()

empenhos_2025 = empenhos_2025.merge(
    secretarias_identificadas[["Código", "Secretaria"]],
    on="Código",
    how="left",
    validate="one_to_one"
)

display(
    empenhos_2025[
        ["Código", "Secretaria", "Julho 2025", "Dezembro 2025"]
    ].style.format({
        "Julho 2025": "R$ {:,.2f}",
        "Dezembro 2025": "R$ {:,.2f}"
    })
)

print("\nCONFERÊNCIA: OK")


Envie os balancetes de JULHO e DEZEMBRO de 2025.


Saving f470e23e59ea630e4913a435de0322fe.pdf to f470e23e59ea630e4913a435de0322fe.pdf
Saving d973eaf5fd887965508b6373.pdf to d973eaf5fd887965508b6373.pdf
Julho 2025: R$ 183,545,888.90
Dezembro 2025: R$ 280,532,572.59


,Código,Secretaria,Julho 2025,Dezembro 2025
0,02.01,Administração e Finanças,"R$ 28,199,559.23","R$ 49,243,676.96"
1,02.02,Educação e Cultura,"R$ 53,386,671.30","R$ 94,172,910.67"
2,02.03,Saúde e Promoção Social,"R$ 75,220,439.01","R$ 96,089,920.94"
3,02.04,"Obras, Viação e Serviços","R$ 9,967,904.04","R$ 13,697,288.82"
4,02.05,Agricultura e Meio Ambiente,"R$ 4,640,909.48","R$ 7,165,642.41"
5,02.06,"Trânsito, Segurança e Defesa Civil","R$ 6,219,448.83","R$ 10,937,683.21"
6,02.07,"Esportes, Turismo, Lazer e Juventude","R$ 3,772,578.19","R$ 5,604,426.06"
7,02.08,Trabalho e Desenvolvimento Econômico,"R$ 2,138,378.82","R$ 3,621,023.52"



CONFERÊNCIA: OK


In [11]:

# ETAPA 10 — PROJEÇÃO DE EMPENHOS DE 2026 POR SECRETARIA
# Não altera as tabelas originais.

import pandas as pd
import numpy as np

projecao_secretarias = comparativo_secretarias.merge(
    empenhos_2025[
        ["Código", "Julho 2025", "Dezembro 2025"]
    ],
    on="Código",
    how="left",
    validate="one_to_one"
)

# Verificar dados históricos
assert projecao_secretarias["Julho 2025"].notna().all()
assert projecao_secretarias["Dezembro 2025"].notna().all()
assert (projecao_secretarias["Julho 2025"] > 0).all()

# Fator de sazonalidade observado em 2025
projecao_secretarias["Fator sazonal"] = (
    projecao_secretarias["Dezembro 2025"]
    / projecao_secretarias["Julho 2025"]
)

# Projetar fechamento de 2026
projecao_secretarias["Projeção empenhado dez/2026"] = (
    projecao_secretarias["Empenhado até julho"]
    * projecao_secretarias["Fator sazonal"]
)

# Comparar orçamento 2027 com projeção 2026
projecao_secretarias["Diferença vs projeção R$"] = (
    projecao_secretarias["LOA proposta 2027"]
    - projecao_secretarias["Projeção empenhado dez/2026"]
)

projecao_secretarias["Diferença vs projeção %"] = (
    projecao_secretarias["Diferença vs projeção R$"]
    / projecao_secretarias["Projeção empenhado dez/2026"]
) * 100

colunas = [
    "Secretaria",
    "Fator sazonal",
    "Projeção empenhado dez/2026",
    "LOA proposta 2027",
    "Diferença vs projeção R$",
    "Diferença vs projeção %"
]

display(
    projecao_secretarias[colunas].style.format({
        "Fator sazonal": "{:.3f}",
        "Projeção empenhado dez/2026": "R$ {:,.2f}",
        "LOA proposta 2027": "R$ {:,.2f}",
        "Diferença vs projeção R$": "R$ {:+,.2f}",
        "Diferença vs projeção %": "{:+.2f}%"
    })
)

print("\nTOTAIS")
print(
    "Empenhado jul/2026:",
    f"R$ {projecao_secretarias['Empenhado até julho'].sum():,.2f}"
)
print(
    "Projeção de empenhos 2026:",
    f"R$ {projecao_secretarias['Projeção empenhado dez/2026'].sum():,.2f}"
)
print(
    "LOA proposta 2027:",
    f"R$ {projecao_secretarias['LOA proposta 2027'].sum():,.2f}"
)
print(
    "Diferença vs projeção:",
    f"R$ {projecao_secretarias['Diferença vs projeção R$'].sum():,.2f}"
)

# Identificar projeções acima da dotação atualizada em julho
alertas = projecao_secretarias.loc[
    projecao_secretarias["Projeção empenhado dez/2026"]
    > projecao_secretarias["Dotação atualizada"],
    ["Secretaria", "Dotação atualizada", "Projeção empenhado dez/2026"]
]

print("\nSECRETARIAS COM PROJEÇÃO ACIMA DA DOTAÇÃO DE JULHO:")
display(alertas)


,Secretaria,Fator sazonal,Projeção empenhado dez/2026,LOA proposta 2027,Diferença vs projeção R$,Diferença vs projeção %
0,Administração e Finanças,1.746,"R$ 77,006,463.58","R$ 95,480,000.00","R$ +18,473,536.42",+23.99%
1,Educação e Cultura,1.764,"R$ 99,468,225.11","R$ 96,410,000.00","R$ -3,058,225.11",-3.07%
2,Saúde e Promoção Social,1.277,"R$ 81,490,072.93","R$ 85,305,000.00","R$ +3,814,927.07",+4.68%
3,"Obras, Viação e Serviços",1.374,"R$ 21,830,271.41","R$ 16,150,000.00","R$ -5,680,271.41",-26.02%
4,Agricultura e Meio Ambiente,1.544,"R$ 6,960,388.76","R$ 9,273,000.00","R$ +2,312,611.24",+33.23%
5,"Trânsito, Segurança e Defesa Civil",1.759,"R$ 11,194,810.46","R$ 13,016,000.00","R$ +1,821,189.54",+16.27%
6,"Esportes, Turismo, Lazer e Juventude",1.486,"R$ 5,062,278.01","R$ 4,601,000.00","R$ -461,278.01",-9.11%
7,Trabalho e Desenvolvimento Econômico,1.693,"R$ 3,488,768.02","R$ 3,980,000.00","R$ +491,231.98",+14.08%



TOTAIS
Empenhado jul/2026: R$ 196,506,138.54
Projeção de empenhos 2026: R$ 306,501,278.30
LOA proposta 2027: R$ 324,215,000.00
Diferença vs projeção: R$ 17,713,721.70

SECRETARIAS COM PROJEÇÃO ACIMA DA DOTAÇÃO DE JULHO:


,Secretaria,Dotação atualizada,Projeção empenhado dez/2026
0,Administração e Finanças,74100300.00,7.700646e+07
1,Educação e Cultura,96269225.95,9.946823e+07
2,Saúde e Promoção Social,81363147.73,8.149007e+07
3,"Obras, Viação e Serviços",20671253.63,2.183027e+07


In [14]:

# ETAPA 11 — ADMINISTRAÇÃO X FAPEN (CORRIGIDA)
# Não modifica as bases anteriores.

import pdfplumber
import pandas as pd
import re

arquivos = {
    "Julho 2025": "d973eaf5fd887965508b6373.pdf",
    "Dezembro 2025": "f470e23e59ea630e4913a435de0322fe.pdf",
    "Julho 2026": "394d9791d49484b888a207f2d5afb3c3.pdf"
}

padrao_valor = r"-?[\d.]+,\d{2}"
registros = []

for periodo, arquivo in arquivos.items():

    unidade = None

    with pdfplumber.open(arquivo) as pdf:

        for pagina, folha in enumerate(pdf.pages, start=1):

            texto = folha.extract_text() or ""

            for linha in texto.splitlines():

                # Identificar QUALQUER unidade executora
                identificacao = re.match(
                    r"^\s*(\d{2}\.\d{2}\.\d{2})\s+",
                    linha
                )

                if identificacao:
                    unidade = identificacao.group(1)

                # Extrair apenas os totais das duas unidades
                if "Total por Unidade Executora" in linha:

                    if unidade not in ["02.01.01", "02.01.02"]:
                        continue

                    trecho = linha.split(
                        "Total por Unidade Executora", 1
                    )[1]

                    valores = re.findall(padrao_valor, trecho)

                    if len(valores) != 16:
                        raise ValueError(
                            f"Erro na página {pagina} de {arquivo}"
                        )

                    registros.append({
                        "Período": periodo,
                        "Código": unidade,
                        "Unidade": (
                            "Administração"
                            if unidade == "02.01.01"
                            else "FAPEN"
                        ),
                        "Empenhado": float(
                            valores[6]
                            .replace(".", "")
                            .replace(",", ".")
                        ),
                        "Página PDF": pagina
                    })

resultado_fapen = pd.DataFrame(registros)

# Conferir registros ANTES de criar a tabela
print("REGISTROS IDENTIFICADOS:")
display(resultado_fapen)

print("\nTOTAL DE REGISTROS:", len(resultado_fapen))

# Verificar duplicidades antes do pivot
duplicados = resultado_fapen.duplicated(
    subset=["Período", "Código"],
    keep=False
)

if duplicados.any():
    print("\nATENÇÃO: REGISTROS DUPLICADOS")
    display(resultado_fapen.loc[duplicados])
    raise ValueError("Duplicidades identificadas. Conferir PDF.")

assert len(resultado_fapen) == 6, (
    f"Esperados 6 registros, encontrados {len(resultado_fapen)}"
)

# Agora montar a tabela
tabela_fapen = resultado_fapen.pivot(
    index=["Código", "Unidade"],
    columns="Período",
    values="Empenhado"
).reset_index()

display(
    tabela_fapen.style.format({
        "Julho 2025": "R$ {:,.2f}",
        "Dezembro 2025": "R$ {:,.2f}",
        "Julho 2026": "R$ {:,.2f}"
    })
)

# Conferências com o balancete
controles = {
    "Julho 2025": 28199559.23,
    "Dezembro 2025": 49243676.96,
    "Julho 2026": 44098013.49
}

for periodo, esperado in controles.items():

    total = tabela_fapen[periodo].sum()

    print(
        f"{periodo}: R$ {total:,.2f}"
    )

    assert round(total, 2) == round(esperado, 2), (
        f"Divergência no período {periodo}"
    )

print("\nCONFERÊNCIA: OK")


REGISTROS IDENTIFICADOS:


,Período,Código,Unidade,Empenhado,Página PDF
0,Julho 2025,02.01.01,Administração,8180799.55,2
1,Julho 2025,02.01.02,FAPEN,20018759.68,2
2,Dezembro 2025,02.01.01,Administração,11452437.95,2
3,Dezembro 2025,02.01.02,FAPEN,37791239.01,2
4,Julho 2026,02.01.01,Administração,21987454.32,2
5,Julho 2026,02.01.02,FAPEN,22110559.17,3



TOTAL DE REGISTROS: 6


Período,Código,Unidade,Dezembro 2025,Julho 2025,Julho 2026
0,02.01.01,Administração,"R$ 11,452,437.95","R$ 8,180,799.55","R$ 21,987,454.32"
1,02.01.02,FAPEN,"R$ 37,791,239.01","R$ 20,018,759.68","R$ 22,110,559.17"


Julho 2025: R$ 28,199,559.23
Dezembro 2025: R$ 49,243,676.96
Julho 2026: R$ 44,098,013.49

CONFERÊNCIA: OK


In [15]:

# ETAPA 12 — PROJEÇÃO SEPARADA: ADMINISTRAÇÃO E FAPEN
# Preserva todas as bases originais.

projecao_fapen = tabela_fapen.copy()

projecao_fapen["Fator sazonal"] = (
    projecao_fapen["Dezembro 2025"]
    / projecao_fapen["Julho 2025"]
)

projecao_fapen["Projeção 2026"] = (
    projecao_fapen["Julho 2026"]
    * projecao_fapen["Fator sazonal"]
)

display(
    projecao_fapen[
        [
            "Unidade",
            "Fator sazonal",
            "Julho 2026",
            "Projeção 2026"
        ]
    ].style.format({
        "Fator sazonal": "{:.3f}",
        "Julho 2026": "R$ {:,.2f}",
        "Projeção 2026": "R$ {:,.2f}"
    })
)

projecao_separada = projecao_fapen["Projeção 2026"].sum()

projecao_anterior = projecao_secretarias.loc[
    projecao_secretarias["Código"] == "02.01",
    "Projeção empenhado dez/2026"
].iloc[0]

print("\nCOMPARAÇÃO DE MÉTODOS")

print(
    "Projeção anterior (conjunta):",
    f"R$ {projecao_anterior:,.2f}"
)

print(
    "Projeção separada:",
    f"R$ {projecao_separada:,.2f}"
)

print(
    "Diferença entre métodos:",
    f"R$ {projecao_separada - projecao_anterior:+,.2f}"
)

print(
    "\nLOA 2027 de Administração e Finanças:",
    "R$ 95,480,000.00"
)


Período,Unidade,Fator sazonal,Julho 2026,Projeção 2026
0,Administração,1.400,"R$ 21,987,454.32","R$ 30,780,604.60"
1,FAPEN,1.888,"R$ 22,110,559.17","R$ 41,740,119.75"



COMPARAÇÃO DE MÉTODOS
Projeção anterior (conjunta): R$ 77,006,463.58
Projeção separada: R$ 72,520,724.35
Diferença entre métodos: R$ -4,485,739.23

LOA 2027 de Administração e Finanças: R$ 95,480,000.00


In [16]:

# ETAPA 13 — COMPARAÇÃO DA ESTRUTURA ORÇAMENTÁRIA
# Apenas leitura dos balancetes; sem projeções.

import pdfplumber
import pandas as pd
import re

arquivos_comparacao = {
    "Julho 2025": "d973eaf5fd887965508b6373.pdf",
    "Julho 2026": "394d9791d49484b888a207f2d5afb3c3.pdf"
}

registros_estrutura = []

for periodo, arquivo in arquivos_comparacao.items():
    unidade = None

    with pdfplumber.open(arquivo) as pdf:
        for pagina, folha in enumerate(pdf.pages, 1):
            for linha in (folha.extract_text() or "").splitlines():

                identificacao = re.match(
                    r"^\s*(\d{2}\.\d{2}\.\d{2})\s+",
                    linha
                )

                if identificacao:
                    unidade = identificacao.group(1)

                if "Total por Unidade Executora" not in linha:
                    continue

                if unidade not in ["02.01.01", "02.01.02"]:
                    continue

                trecho = linha.split(
                    "Total por Unidade Executora", 1
                )[1]

                valores = re.findall(
                    r"-?[\d.]+,\d{2}", trecho
                )

                assert len(valores) == 16

                def converter(texto):
                    return float(
                        texto.replace(".", "").replace(",", ".")
                    )

                registros_estrutura.append({
                    "Período": periodo,
                    "Unidade": (
                        "Administração"
                        if unidade == "02.01.01"
                        else "FAPEN"
                    ),
                    "Dotação inicial": converter(valores[0]),
                    "Dotação atualizada": converter(valores[2]),
                    "Empenhado acumulado": converter(valores[6])
                })

estrutura = pd.DataFrame(registros_estrutura)

assert len(estrutura) == 4
assert not estrutura.duplicated(
    ["Período", "Unidade"]
).any()

display(
    estrutura.style.format({
        "Dotação inicial": "R$ {:,.2f}",
        "Dotação atualizada": "R$ {:,.2f}",
        "Empenhado acumulado": "R$ {:,.2f}"
    })
)

print("\nCONFERÊNCIA: 4 REGISTROS")


,Período,Unidade,Dotação inicial,Dotação atualizada,Empenhado acumulado
0,Julho 2025,Administração,"R$ 10,800,000.00","R$ 11,424,265.63","R$ 8,180,799.55"
1,Julho 2025,FAPEN,"R$ 34,750,000.00","R$ 34,750,000.00","R$ 20,018,759.68"
2,Julho 2026,Administração,"R$ 33,065,000.00","R$ 33,810,300.00","R$ 21,987,454.32"
3,Julho 2026,FAPEN,"R$ 40,290,000.00","R$ 40,290,000.00","R$ 22,110,559.17"



CONFERÊNCIA: 4 REGISTROS


In [17]:

# ETAPA 14 — LOCALIZAR APORTE PREVIDENCIÁRIO
# Apenas consulta os PDFs; não altera projeções.

import pdfplumber

arquivos_aporte = {
    "Julho 2025": "d973eaf5fd887965508b6373.pdf",
    "Dezembro 2025": "f470e23e59ea630e4913a435de0322fe.pdf",
    "Julho 2026": "394d9791d49484b888a207f2d5afb3c3.pdf"
}

for periodo, arquivo in arquivos_aporte.items():
    print(f"\n--- {periodo} ---")
    encontrados = []

    with pdfplumber.open(arquivo) as pdf:
        for numero, pagina in enumerate(pdf.pages, 1):
            texto = pagina.extract_text() or ""

            for linha in texto.splitlines():
                if "3.3.91.97" in linha:
                    encontrados.append((numero, linha.strip()))

    if encontrados:
        for pagina, linha in encontrados:
            print(f"Página {pagina}: {linha}")
    else:
        print("Classificação 3.3.91.97 não localizada.")



--- Julho 2025 ---
Página 13: 3.3.91.97APORTE PARA COBERTURA DO DÉFICIT 1 118 9.385.000,00 0,00 9.385.000,00 4.604.827,12 4.097.291,04 682.881,84 4.780.172,88 0,00 4.097.291,04 682.881,84 4.780.172,88 0,00 4.097.291,04 682.881,84 4.780.172,88 0,00

--- Dezembro 2025 ---
Página 15: 3.3.91.97APORTE PARA COBERTURA DO DÉFICIT 1 118 9.385.000,00 -1.190.415,00 8.194.585,00 2,89 7.511.700,24 682.881,87 8.194.582,11 0,00 7.511.700,24 682.881,87 8.194.582,11 0,00 7.511.700,24 682.881,87 8.194.582,11 0,00

--- Julho 2026 ---
Página 2: 3.3.91.97APORTE PARA COBERTURA DO DÉFICIT 1 14 13.077.000,00 0,00 13.077.000,00 5.448.750,00 6.538.500,00 1.089.750,00 7.628.250,00 0,00 6.538.500,00 1.089.750,00 7.628.250,00 0,00 6.538.500,00 1.089.750,00 7.628.250,00 0,00


In [18]:

# ETAPA 15 — IDENTIFICAR A LOCALIZAÇÃO DO APORTE
# Apenas consulta os documentos originais.

import pdfplumber

arquivos_aporte = {
    "Julho 2025": "d973eaf5fd887965508b6373.pdf",
    "Dezembro 2025": "f470e23e59ea630e4913a435de0322fe.pdf",
    "Julho 2026": "394d9791d49484b888a207f2d5afb3c3.pdf"
}

for periodo, arquivo in arquivos_aporte.items():

    print(f"\n{'=' * 50}")
    print(periodo)
    print("=" * 50)

    with pdfplumber.open(arquivo) as pdf:
        for numero, pagina in enumerate(pdf.pages, start=1):

            linhas = (pagina.extract_text() or "").splitlines()

            for i, linha in enumerate(linhas):

                if "3.3.91.97" in linha:

                    print(f"\nPágina {numero}")
                    print("CONTEXTO DA DESPESA:\n")

                    inicio = max(0, i - 15)

                    for contexto in linhas[inicio:i + 1]:
                        print(contexto)



Julho 2025

Página 13
CONTEXTO DA DESPESA:

02.03.02 AÇÃO SOCIAL
08.244.0004.2004MANUTENÇÃO DO SETOR
3.3.90.39OUTROS SERV.TERCEIROS-PESSOA JURÍDICA 1 113 5.300.000,00 0,00 5.300.000,00 0,00 2.698.500,00 2.601.500,00 5.300.000,00 2.148.500,00 2.698.500,00 453.000,00 3.151.500,00 2.148.500,00 2.698.500,00 453.000,00 3.151.500,00 0,00
99OUTROS SERVIÇOS DE TERCEIROS - PESSOA 2.698.500,00 2.601.500,00 5.300.000,00 2.148.500,00 2.698.500,00 453.000,00 3.151.500,00 2.148.500,00 2.698.500,00 453.000,00 3.151.500,00 0,00
09.271.0004.0002ENCARGOS SOCIAIS
3.1.90.03PENSÕES 1 114 360.000,00 0,00 360.000,00 165.473,56 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00
01PESSOAL CIVIL 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00
3.1.90.13OBRIGAÇÕES PATRONAIS 1 115 55.000,00 0,00 55.000,00 20.535,03 29.076,44 5.388,53 34.464,97 5.388,53 29.076,44 5.388,53 34.464,97 0,00 29.076

In [19]:

# ETAPA 16 — ISOLAR O APORTE PREVIDENCIÁRIO
# Não altera nenhuma base original.

import pandas as pd

# Valores extraídos dos balancetes oficiais
aportes = pd.DataFrame([
    {
        "Período": "Julho 2025",
        "Código": "02.03",
        "Aporte empenhado": 4780172.88
    },
    {
        "Período": "Dezembro 2025",
        "Código": "02.03",
        "Aporte empenhado": 8194582.11
    },
    {
        "Período": "Julho 2026",
        "Código": "02.01",
        "Aporte empenhado": 7628250.00
    }
])

# Preservar valores de origem
base_ajuste = historico_2025[
    ["Período", "Código", "Empenhado acumulado"]
].copy()

base_2026 = secretarias_identificadas[
    ["Código", "Empenhado até julho"]
].copy()

base_2026["Período"] = "Julho 2026"
base_2026 = base_2026.rename(columns={
    "Empenhado até julho": "Empenhado acumulado"
})

base_ajuste = pd.concat(
    [base_ajuste, base_2026],
    ignore_index=True
)

# Associar aportes somente às unidades de origem
base_ajuste = base_ajuste.merge(
    aportes,
    on=["Período", "Código"],
    how="left",
    validate="one_to_one"
)

base_ajuste["Aporte empenhado"] = (
    base_ajuste["Aporte empenhado"].fillna(0)
)

# Retirar aporte apenas para comparação analítica
base_ajuste["Empenhado sem aporte"] = (
    base_ajuste["Empenhado acumulado"]
    - base_ajuste["Aporte empenhado"]
)

# Conferências
assert (base_ajuste["Empenhado sem aporte"] >= 0).all()

assert (
    (
        base_ajuste["Empenhado sem aporte"]
        + base_ajuste["Aporte empenhado"]
        - base_ajuste["Empenhado acumulado"]
    ).abs().max() < 0.01
)

# Mostrar somente as duas secretarias afetadas
display(
    base_ajuste.loc[
        base_ajuste["Código"].isin(["02.01", "02.03"]),
        [
            "Período",
            "Código",
            "Empenhado acumulado",
            "Aporte empenhado",
            "Empenhado sem aporte"
        ]
    ].style.format({
        "Empenhado acumulado": "R$ {:,.2f}",
        "Aporte empenhado": "R$ {:,.2f}",
        "Empenhado sem aporte": "R$ {:,.2f}"
    })
)

print("\nCONFERÊNCIA: aportes separados sem alterar os totais.")


,Período,Código,Empenhado acumulado,Aporte empenhado,Empenhado sem aporte
0,Dezembro 2025,02.01,"R$ 49,243,676.96",R$ 0.00,"R$ 49,243,676.96"
2,Dezembro 2025,02.03,"R$ 96,089,920.94","R$ 8,194,582.11","R$ 87,895,338.83"
8,Julho 2025,02.01,"R$ 28,199,559.23",R$ 0.00,"R$ 28,199,559.23"
10,Julho 2025,02.03,"R$ 75,220,439.01","R$ 4,780,172.88","R$ 70,440,266.13"
16,Julho 2026,02.01,"R$ 44,098,013.49","R$ 7,628,250.00","R$ 36,469,763.49"
18,Julho 2026,02.03,"R$ 63,791,488.23",R$ 0.00,"R$ 63,791,488.23"



CONFERÊNCIA: aportes separados sem alterar os totais.


In [20]:

# ETAPA 17 — CONFERIR ENCARGOS SOCIAIS
# Apenas leitura dos PDFs, sem alterar nenhuma base.

import pdfplumber
import re

arquivos_encargos = {
    "Julho 2025": "d973eaf5fd887965508b6373.pdf",
    "Dezembro 2025": "f470e23e59ea630e4913a435de0322fe.pdf",
    "Julho 2026": "394d9791d49484b888a207f2d5afb3c3.pdf"
}

rubricas = [
    "3.1.90.03",  # Pensões
    "3.1.90.13",  # Obrigações patronais
    "3.3.90.08",  # Benefícios assistenciais
    "3.3.90.47",  # Obrigações tributárias
    "3.3.91.97"   # Aporte previdenciário
]

for periodo, arquivo in arquivos_encargos.items():

    print(f"\n{'=' * 55}")
    print(periodo)
    print("=" * 55)

    # Identificar o programa correspondente
    programa_alvo = (
        "09.271.0004.0002"
        if "2025" in periodo
        else "09.271.0002.0002"
    )

    with pdfplumber.open(arquivo) as pdf:

        dentro_programa = False

        for pagina, folha in enumerate(pdf.pages, start=1):

            linhas = (folha.extract_text() or "").splitlines()

            for linha in linhas:

                # Uma nova ação encerra a anterior
                acao = re.match(
                    r"^\s*(\d{2}\.\d{3}\.\d{4}\.\d{4})",
                    linha
                )

                if acao:
                    dentro_programa = (
                        acao.group(1) == programa_alvo
                    )

                if not dentro_programa:
                    continue

                if any(
                    linha.strip().startswith(codigo)
                    for codigo in rubricas
                ):
                    print(
                        f"Página {pagina}: {linha.strip()}"
                    )



Julho 2025
Página 13: 3.1.90.03PENSÕES 1 114 360.000,00 0,00 360.000,00 165.473,56 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00 166.905,66 27.620,78 194.526,44 0,00
Página 13: 3.1.90.13OBRIGAÇÕES PATRONAIS 1 115 55.000,00 0,00 55.000,00 20.535,03 29.076,44 5.388,53 34.464,97 5.388,53 29.076,44 5.388,53 34.464,97 0,00 29.076,44 0,00 29.076,44 5.388,53
Página 13: 3.3.90.08OUTROS BENEFÍCIOS ASSISTENCIAIS 1 116 65.000,00 0,00 65.000,00 12.163,26 39.726,15 13.110,59 52.836,74 0,00 39.726,15 13.110,59 52.836,74 0,00 37.133,34 15.703,40 52.836,74 0,00
Página 13: 3.3.90.47OBRIGAÇÕES TRIBUTÁRIAS E 1 117 3.000.000,00 0,00 3.000.000,00 1.388.749,58 1.423.298,06 187.952,36 1.611.250,42 1.113,18 1.423.298,06 187.952,36 1.611.250,42 0,00 1.423.298,06 186.839,18 1.610.137,24 1.113,18
Página 13: 3.3.91.97APORTE PARA COBERTURA DO DÉFICIT 1 118 9.385.000,00 0,00 9.385.000,00 4.604.827,12 4.097.291,04 682.881,84 4.780.172,88 0,00 4.097.291,04 682.881,84 4.780.172,88 0,00 4.0

In [21]:

# ETAPA 18 — RECLASSIFICAÇÃO ANALÍTICA DO HISTÓRICO
# Preserva integralmente os valores originais.
# Não modifica os balancetes nem as bases anteriores.

import pandas as pd

# Totais das cinco rubricas, conforme Etapa 17
encargos_transferidos = {
    "Julho 2025": 6673251.45,
    "Dezembro 2025": 11371042.14
}

# Retomar os empenhos originais das três datas
base_comparavel = base_ajuste[
    ["Período", "Código", "Empenhado acumulado"]
].copy()

base_comparavel["Ajuste analítico"] = 0.0

for periodo, valor in encargos_transferidos.items():

    # Adicionar os encargos ao histórico da Administração
    base_comparavel.loc[
        (base_comparavel["Período"] == periodo)
        & (base_comparavel["Código"] == "02.01"),
        "Ajuste analítico"
    ] = valor

    # Retirar o mesmo valor do histórico da Saúde
    base_comparavel.loc[
        (base_comparavel["Período"] == periodo)
        & (base_comparavel["Código"] == "02.03"),
        "Ajuste analítico"
    ] = -valor

base_comparavel["Empenhado comparável"] = (
    base_comparavel["Empenhado acumulado"]
    + base_comparavel["Ajuste analítico"]
)

# Conferir que nenhum total municipal foi alterado
conferencia = base_comparavel.groupby("Período")[
    ["Empenhado acumulado", "Empenhado comparável"]
].sum()

assert (
    conferencia["Empenhado acumulado"]
    - conferencia["Empenhado comparável"]
).abs().max() < 0.01

# Mostrar somente as secretarias afetadas
resultado_comparavel = base_comparavel.loc[
    base_comparavel["Código"].isin(["02.01", "02.03"])
].copy()

resultado_comparavel["Secretaria"] = resultado_comparavel[
    "Código"
].map({
    "02.01": "Administração e Finanças",
    "02.03": "Saúde e Promoção Social"
})

display(
    resultado_comparavel[
        [
            "Período",
            "Secretaria",
            "Empenhado acumulado",
            "Ajuste analítico",
            "Empenhado comparável"
        ]
    ].sort_values(
        ["Período", "Secretaria"]
    ).style.format({
        "Empenhado acumulado": "R$ {:,.2f}",
        "Ajuste analítico": "R$ {:+,.2f}",
        "Empenhado comparável": "R$ {:,.2f}"
    })
)

print("\nCONFERÊNCIA DOS TOTAIS:")
display(conferencia)

print("\nRECLASSIFICAÇÃO CONCLUÍDA SEM ALTERAR TOTAIS.")


,Período,Secretaria,Empenhado acumulado,Ajuste analítico,Empenhado comparável
0,Dezembro 2025,Administração e Finanças,"R$ 49,243,676.96","R$ +11,371,042.14","R$ 60,614,719.10"
2,Dezembro 2025,Saúde e Promoção Social,"R$ 96,089,920.94","R$ -11,371,042.14","R$ 84,718,878.80"
8,Julho 2025,Administração e Finanças,"R$ 28,199,559.23","R$ +6,673,251.45","R$ 34,872,810.68"
10,Julho 2025,Saúde e Promoção Social,"R$ 75,220,439.01","R$ -6,673,251.45","R$ 68,547,187.56"
16,Julho 2026,Administração e Finanças,"R$ 44,098,013.49",R$ +0.00,"R$ 44,098,013.49"
18,Julho 2026,Saúde e Promoção Social,"R$ 63,791,488.23",R$ +0.00,"R$ 63,791,488.23"



CONFERÊNCIA DOS TOTAIS:


,Empenhado acumulado,Empenhado comparável
Período,,
Dezembro 2025,2.805326e+08,2.805326e+08
Julho 2025,1.835459e+08,1.835459e+08
Julho 2026,1.965061e+08,1.965061e+08



RECLASSIFICAÇÃO CONCLUÍDA SEM ALTERAR TOTAIS.


In [22]:

# ETAPA 19 — PROJEÇÃO 2026 COM HISTÓRICO RECLASSIFICADO
# Não altera as tabelas ou os valores originais.

import pandas as pd

# Organizar os valores comparáveis por período
historico_ajustado = base_comparavel.pivot(
    index="Código",
    columns="Período",
    values="Empenhado comparável"
).reset_index()

# Incorporar os nomes e a proposta de 2027
projecao_ajustada = (
    comparativo_secretarias[
        [
            "Código",
            "Secretaria",
            "Dotação atualizada",
            "LOA proposta 2027"
        ]
    ]
    .merge(
        historico_ajustado,
        on="Código",
        how="left",
        validate="one_to_one"
    )
)

assert len(projecao_ajustada) == 8
assert projecao_ajustada[
    ["Julho 2025", "Dezembro 2025", "Julho 2026"]
].notna().all().all()
assert (projecao_ajustada["Julho 2025"] > 0).all()

# Sazonalidade histórica ajustada
projecao_ajustada["Fator ajustado"] = (
    projecao_ajustada["Dezembro 2025"]
    / projecao_ajustada["Julho 2025"]
)

# Projeção de fechamento de 2026
projecao_ajustada["Projeção 2026 ajustada"] = (
    projecao_ajustada["Julho 2026"]
    * projecao_ajustada["Fator ajustado"]
)

# Comparação com a proposta de 2027
projecao_ajustada["Diferença R$"] = (
    projecao_ajustada["LOA proposta 2027"]
    - projecao_ajustada["Projeção 2026 ajustada"]
)

projecao_ajustada["Diferença %"] = (
    projecao_ajustada["Diferença R$"]
    / projecao_ajustada["Projeção 2026 ajustada"]
) * 100

# Exibir resultados
colunas = [
    "Secretaria",
    "Fator ajustado",
    "Projeção 2026 ajustada",
    "LOA proposta 2027",
    "Diferença R$",
    "Diferença %"
]

display(
    projecao_ajustada[colunas].style.format({
        "Fator ajustado": "{:.3f}",
        "Projeção 2026 ajustada": "R$ {:,.2f}",
        "LOA proposta 2027": "R$ {:,.2f}",
        "Diferença R$": "R$ {:+,.2f}",
        "Diferença %": "{:+.2f}%"
    })
)

# Comparação dos cenários
total_original = (
    projecao_secretarias[
        "Projeção empenhado dez/2026"
    ].sum()
)

total_ajustado = (
    projecao_ajustada[
        "Projeção 2026 ajustada"
    ].sum()
)

total_2027 = (
    projecao_ajustada["LOA proposta 2027"].sum()
)

print("\nCOMPARAÇÃO DOS CENÁRIOS")
print(f"Projeção original: R$ {total_original:,.2f}")
print(f"Projeção ajustada: R$ {total_ajustado:,.2f}")
print(f"Diferença entre métodos: R$ {total_ajustado-total_original:+,.2f}")

print("\nCOMPARAÇÃO COM A LOA 2027")
print(f"LOA proposta 2027: R$ {total_2027:,.2f}")
print(f"Diferença vs projeção ajustada: R$ {total_2027-total_ajustado:+,.2f}")
print(f"Variação: {(total_2027/total_ajustado-1)*100:+.2f}%")

# Identificar projeções acima das dotações de julho
print("\nPROJEÇÕES ACIMA DA DOTAÇÃO ATUALIZADA:")
alertas_ajustados = projecao_ajustada.loc[
    projecao_ajustada["Projeção 2026 ajustada"]
    > projecao_ajustada["Dotação atualizada"],
    [
        "Secretaria",
        "Dotação atualizada",
        "Projeção 2026 ajustada"
    ]
]

display(
    alertas_ajustados.style.format({
        "Dotação atualizada": "R$ {:,.2f}",
        "Projeção 2026 ajustada": "R$ {:,.2f}"
    })
)


,Secretaria,Fator ajustado,Projeção 2026 ajustada,LOA proposta 2027,Diferença R$,Diferença %
0,Administração e Finanças,1.738,"R$ 76,649,649.07","R$ 95,480,000.00","R$ +18,830,350.93",+24.57%
1,Educação e Cultura,1.764,"R$ 99,468,225.11","R$ 96,410,000.00","R$ -3,058,225.11",-3.07%
2,Saúde e Promoção Social,1.236,"R$ 78,841,212.20","R$ 85,305,000.00","R$ +6,463,787.80",+8.20%
3,"Obras, Viação e Serviços",1.374,"R$ 21,830,271.41","R$ 16,150,000.00","R$ -5,680,271.41",-26.02%
4,Agricultura e Meio Ambiente,1.544,"R$ 6,960,388.76","R$ 9,273,000.00","R$ +2,312,611.24",+33.23%
5,"Trânsito, Segurança e Defesa Civil",1.759,"R$ 11,194,810.46","R$ 13,016,000.00","R$ +1,821,189.54",+16.27%
6,"Esportes, Turismo, Lazer e Juventude",1.486,"R$ 5,062,278.01","R$ 4,601,000.00","R$ -461,278.01",-9.11%
7,Trabalho e Desenvolvimento Econômico,1.693,"R$ 3,488,768.02","R$ 3,980,000.00","R$ +491,231.98",+14.08%



COMPARAÇÃO DOS CENÁRIOS
Projeção original: R$ 306,501,278.30
Projeção ajustada: R$ 303,495,603.05
Diferença entre métodos: R$ -3,005,675.25

COMPARAÇÃO COM A LOA 2027
LOA proposta 2027: R$ 324,215,000.00
Diferença vs projeção ajustada: R$ +20,719,396.95
Variação: +6.83%

PROJEÇÕES ACIMA DA DOTAÇÃO ATUALIZADA:


,Secretaria,Dotação atualizada,Projeção 2026 ajustada
0,Administração e Finanças,"R$ 74,100,300.00","R$ 76,649,649.07"
1,Educação e Cultura,"R$ 96,269,225.95","R$ 99,468,225.11"
3,"Obras, Viação e Serviços","R$ 20,671,253.63","R$ 21,830,271.41"


In [23]:

# ETAPA 20 — EXCEL AUDITÁVEL POR SECRETARIA
# Não altera os dados originais nem sobrescreve o Excel por função.

import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.workbook.properties import CalcProperties
from google.colab import files

# 1. Verificar bases previamente validadas
bases = [
    "historico_2025",
    "secretarias_identificadas",
    "proposta_2027",
    "tabela_fapen"
]

for nome in bases:
    if nome not in globals():
        raise ValueError(f"Base não encontrada: {nome}")

# 2. Organizar SOMENTE os dados originais
h25 = historico_2025.copy()
h26 = secretarias_identificadas.copy()
loa = proposta_2027.copy()
fapen = tabela_fapen.copy()

assert len(h25) == 16
assert len(h26) == 8
assert len(loa) == 8
assert len(fapen) == 2

assert round(h25.loc[
    h25["Período"] == "Julho 2025",
    "Empenhado acumulado"
].sum(), 2) == 183545888.90

assert round(h25.loc[
    h25["Período"] == "Dezembro 2025",
    "Empenhado acumulado"
].sum(), 2) == 280532572.59

assert round(h26["Dotação atualizada"].sum(), 2) == 303992427.31
assert round(h26["Empenhado até julho"].sum(), 2) == 196506138.54
assert round(loa["LOA proposta 2027"].sum(), 2) == 324215000

# 3. Rubricas ORIGINAIS identificadas na etapa 17
# A reclassificação será calculada por fórmulas no Excel.
# Em 2025, origem: 02.03 (Saúde)
# Em 2026, registro: 02.01 (Administração)

rubricas = [
    ("3.1.90.03", "Pensões",
     194526.44, 360776.00, 202591.38),
    ("3.1.90.13", "Obrigações patronais",
     34464.97, 58600.00, 45488.00),
    ("3.3.90.08", "Benefícios assistenciais",
     52836.74, 102041.63, 48051.08),
    ("3.3.90.47", "Obrigações tributárias",
     1611250.42, 2655042.40, 1641148.39),
    ("3.3.91.97", "Aporte ao déficit",
     4780172.88, 8194582.11, 7628250.00)
]

assert round(sum(x[2] for x in rubricas), 2) == 6673251.45
assert round(sum(x[3] for x in rubricas), 2) == 11371042.14
assert round(sum(x[4] for x in rubricas), 2) == 9565528.85

# 4. Criar o arquivo
wb = Workbook()
wb.remove(wb.active)

nomes_abas = [
    "LEIA_ME",
    "HISTORICO_2025",
    "EXECUCAO_2026",
    "LOA_2027",
    "ENCARGOS",
    "COMPARATIVO",
    "FAPEN",
    "CONFERENCIA"
]

abas = {nome: wb.create_sheet(nome) for nome in nomes_abas}

azul = "17324D"
azul_claro = "DCE8F1"
cinza = "F0F3F6"
verde = "E5F2E8"
amarelo = "FFF2CC"

moeda = '"R$" #,##0.00;[Red]("R$" #,##0.00)'
percentual = '0.00%;[Red](0.00%)'
fator = '0.000'

def escrever_cabecalho(ws, titulos):
    ws.append(titulos)
    for celula in ws[1]:
        celula.fill = PatternFill("solid", fgColor=azul)
        celula.font = Font(color="FFFFFF", bold=True)
        celula.alignment = Alignment(wrap_text=True)
    ws.row_dimensions[1].height = 32
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

def finalizar_aba(ws, larguras, colunas_moeda=(),
                  colunas_pct=(), colunas_fator=()):
    for coluna, largura in larguras.items():
        ws.column_dimensions[coluna].width = largura

    for linha in range(2, ws.max_row + 1):
        for celula in ws[linha]:
            if linha % 2 == 0:
                celula.fill = PatternFill(
                    "solid", fgColor="F5F7FA"
                )

        for coluna in colunas_moeda:
            ws[f"{coluna}{linha}"].number_format = moeda
        for coluna in colunas_pct:
            ws[f"{coluna}{linha}"].number_format = percentual
        for coluna in colunas_fator:
            ws[f"{coluna}{linha}"].number_format = fator

# 5. Orientações e metodologia
ws = abas["LEIA_ME"]
orientacoes = [
    ["COMPARATIVO ORÇAMENTÁRIO — CERQUILHO"],
    ["Recorte", "Oito unidades orçamentárias do Executivo"],
    ["Exclusões", "Câmara Municipal e SAAEC"],
    ["Referência 2026", "Balancete de julho de 2026"],
    ["Histórico", "Balancetes de julho e dezembro de 2025"],
    ["2027", "Proposta da LOA 2027 — valores a conferir na fonte"],
    ["Base de comparação", "Empenhos acumulados"],
    ["Cenário original", "Sazonalidade sem reclassificação"],
    ["Cenário comparável", "Cinco encargos de 2025 reclassificados analiticamente"],
    ["Fórmula sazonal", "Empenhos dezembro/2025 ÷ julho/2025"],
    ["Projeção 2026", "Empenhos julho/2026 × fator sazonal"],
    ["Diferença", "LOA 2027 − projeção 2026"],
    ["Ajuste histórico", "Transferência analítica de Saúde para Administração"],
    ["FAPEN", "Informação complementar; já incluída em Administração"],
    ["Reserva de contingência", "Incluída nas dotações, sem projeção própria de empenhos"],
    ["Limitação", "Fatores de 2025 não garantem execução equivalente em 2026"],
    ["Limitação adicional", "Apenas cinco rubricas tiveram reclassificação identificada"],
    ["Atenção", "Projeções são cenários, não execução oficial"],
    ["Atualização", "Ao obter novo balancete, preservar as bases históricas"]
]

for linha in orientacoes:
    ws.append(linha)

ws["A1"].font = Font(size=15, bold=True, color=azul)
ws.column_dimensions["A"].width = 29
ws.column_dimensions["B"].width = 85
ws.freeze_panes = "A2"

# 6. Histórico original de 2025
ws = abas["HISTORICO_2025"]
escrever_cabecalho(ws, [
    "Período", "Código", "Secretaria",
    "Empenhado acumulado", "Página PDF", "Fonte"
])

mapa_nomes = dict(zip(h26["Código"], h26["Secretaria"]))

for _, r in h25.sort_values(
    ["Período", "Código"]
).iterrows():
    ws.append([
        r["Período"],
        str(r["Código"]),
        mapa_nomes[str(r["Código"])],
        float(r["Empenhado acumulado"]),
        int(r["Página PDF"]),
        "Balancete da Despesa 2025 — Poder Executivo"
    ])

finalizar_aba(ws, {
    "A": 19, "B": 13, "C": 43,
    "D": 23, "E": 14, "F": 51
}, ["D"])

# 7. Execução original de julho de 2026
ws = abas["EXECUCAO_2026"]
escrever_cabecalho(ws, [
    "Código", "Secretaria", "Dotação inicial",
    "Alteração líquida", "Dotação atualizada",
    "Empenhado julho/2026", "Página PDF", "Fonte"
])

for _, r in h26.sort_values("Código").iterrows():
    ws.append([
        str(r["Código"]),
        r["Secretaria"],
        float(r["Dotação inicial"]),
        float(r["Alteração líquida"]),
        float(r["Dotação atualizada"]),
        float(r["Empenhado até julho"]),
        int(r["Página PDF"]),
        "Balancete da Despesa — Julho 2026"
    ])

finalizar_aba(ws, {
    "A": 13, "B": 43, "C": 23, "D": 23,
    "E": 23, "F": 25, "G": 14, "H": 40
}, ["C", "D", "E", "F"])

# 8. Proposta da LOA 2027
ws = abas["LOA_2027"]
escrever_cabecalho(ws, [
    "Código", "Secretaria",
    "LOA proposta 2027", "Fonte / observação"
])

for _, r in loa.sort_values("Código").iterrows():
    ws.append([
        str(r["Código"]),
        r["Secretaria"],
        float(r["LOA proposta 2027"]),
        "Projeto da LOA 2027 — transcrição validada pelo total"
    ])

finalizar_aba(ws, {
    "A": 13, "B": 43, "C": 25, "D": 57
}, ["C"])

# 9. Encargos sociais: três períodos, valores originais
ws = abas["ENCARGOS"]
escrever_cabecalho(ws, [
    "Período", "Rubrica", "Descrição",
    "Unidade original", "Empenhado",
    "Página PDF", "Unidade comparável"
])

for codigo, descricao, jul25, dez25, jul26 in rubricas:
    for periodo, valor, origem, pagina, comparavel in [
        ("Julho 2025", jul25, "02.03", 13, "02.01"),
        ("Dezembro 2025", dez25, "02.03", 15, "02.01"),
        ("Julho 2026", jul26, "02.01", 2, "02.01")
    ]:
        ws.append([
            periodo, codigo, descricao,
            origem, valor, pagina, comparavel
        ])

finalizar_aba(ws, {
    "A": 20, "B": 17, "C": 32, "D": 21,
    "E": 23, "F": 14, "G": 23
}, ["E"])

# 10. Comparativo — fórmulas nativas do Excel
ws = abas["COMPARATIVO"]
escrever_cabecalho(ws, [
    "Código", "Secretaria",
    "Emp. jul/2025 original",
    "Emp. dez/2025 original",
    "Encargos jul/2025",
    "Encargos dez/2025",
    "Jul/2025 comparável",
    "Dez/2025 comparável",
    "Emp. jul/2026",
    "Dotação atual jul/2026",
    "LOA 2027",
    "Fator original",
    "Fator comparável",
    "Projeção original 2026",
    "Projeção comparável 2026",
    "Dif. LOA vs projeção R$",
    "Dif. LOA vs projeção %",
    "Dif. LOA vs dotação R$",
    "Dif. LOA vs dotação %",
    "Alerta dotação"
])

for i, codigo in enumerate(sorted(mapa_nomes), start=2):
    ws.cell(i, 1, codigo)
    ws.cell(i, 2, mapa_nomes[codigo])

    ws[f"C{i}"] = (
        f'=SUMIFS(HISTORICO_2025!$D$2:$D$17,'
        f'HISTORICO_2025!$A$2:$A$17,"Julho 2025",'
        f'HISTORICO_2025!$B$2:$B$17,$A{i})'
    )
    ws[f"D{i}"] = (
        f'=SUMIFS(HISTORICO_2025!$D$2:$D$17,'
        f'HISTORICO_2025!$A$2:$A$17,"Dezembro 2025",'
        f'HISTORICO_2025!$B$2:$B$17,$A{i})'
    )

    # Adicionar à Administração, retirar da Saúde.
    # Demais secretarias: ajuste zero.
    sinal = f'IF($A{i}="02.01",1,IF($A{i}="02.03",-1,0))'

    ws[f"E{i}"] = (
        f'=SUMIFS(ENCARGOS!$E$2:$E$16,'
        f'ENCARGOS!$A$2:$A$16,"Julho 2025")'
        f'*{sinal}'
    )
    ws[f"F{i}"] = (
        f'=SUMIFS(ENCARGOS!$E$2:$E$16,'
        f'ENCARGOS!$A$2:$A$16,"Dezembro 2025")'
        f'*{sinal}'
    )

    ws[f"G{i}"] = f"=C{i}+E{i}"
    ws[f"H{i}"] = f"=D{i}+F{i}"

    ws[f"I{i}"] = (
        f'=SUMIF(EXECUCAO_2026!$A$2:$A$9,'
        f'$A{i},EXECUCAO_2026!$F$2:$F$9)'
    )
    ws[f"J{i}"] = (
        f'=SUMIF(EXECUCAO_2026!$A$2:$A$9,'
        f'$A{i},EXECUCAO_2026!$E$2:$E$9)'
    )
    ws[f"K{i}"] = (
        f'=SUMIF(LOA_2027!$A$2:$A$9,'
        f'$A{i},LOA_2027!$C$2:$C$9)'
    )

    ws[f"L{i}"] = f'=IFERROR(D{i}/C{i},0)'
    ws[f"M{i}"] = f'=IFERROR(H{i}/G{i},0)'
    ws[f"N{i}"] = f'=I{i}*L{i}'
    ws[f"O{i}"] = f'=I{i}*M{i}'
    ws[f"P{i}"] = f'=K{i}-O{i}'
    ws[f"Q{i}"] = f'=IFERROR(P{i}/O{i},0)'
    ws[f"R{i}"] = f'=K{i}-J{i}'
    ws[f"S{i}"] = f'=IFERROR(R{i}/J{i},0)'
    ws[f"T{i}"] = (
        f'=IF(O{i}>J{i},"ACIMA DA DOTAÇÃO","OK")'
    )

# Linha de totais
linha_total = 10
ws[f"B{linha_total}"] = "TOTAL DO EXECUTIVO"

for coluna in list("CDEFGHIJK") + list("NOPR"):
    ws[f"{coluna}{linha_total}"] = (
        f"=SUM({coluna}2:{coluna}9)"
    )

ws["Q10"] = '=IFERROR(P10/O10,0)'
ws["S10"] = '=IFERROR(R10/J10,0)'

for celula in ws[linha_total]:
    celula.fill = PatternFill("solid", fgColor=azul_claro)
    celula.font = Font(bold=True, color=azul)

larguras = {"A": 13, "B": 43, "T": 23}
for coluna in "CDEFGHIJKMNOPR":
    larguras[coluna] = 24
for coluna in "LQS":
    larguras[coluna] = 18

finalizar_aba(
    ws, larguras,
    list("CDEFGHIJK") + list("NOPR"),
    ["Q", "S"],
    ["L", "M"]
)
ws.freeze_panes = "C2"

# 11. FAPEN como detalhamento complementar
ws = abas["FAPEN"]
escrever_cabecalho(ws, [
    "Código", "Unidade", "Emp. jul/2025",
    "Emp. dez/2025", "Emp. jul/2026",
    "Fator sazonal", "Projeção 2026"
])

for i, (_, r) in enumerate(
    fapen.sort_values("Código").iterrows(), start=2
):
    ws.append([
        r["Código"],
        r["Unidade"],
        float(r["Julho 2025"]),
        float(r["Dezembro 2025"]),
        float(r["Julho 2026"])
    ])
    ws[f"F{i}"] = f'=IFERROR(D{i}/C{i},0)'
    ws[f"G{i}"] = f"=E{i}*F{i}"

ws["B4"] = "TOTAL ADMINISTRAÇÃO + FAPEN"
for coluna in ["C", "D", "E", "G"]:
    ws[f"{coluna}4"] = f"=SUM({coluna}2:{coluna}3)"

ws["B6"] = "Detalhamento complementar — NÃO somar ao comparativo."
finalizar_aba(
    ws, {
        "A": 15, "B": 48, "C": 24, "D": 24,
        "E": 24, "F": 19, "G": 25
    },
    ["C", "D", "E", "G"], [], ["F"]
)

# 12. Conferências por fórmulas
ws = abas["CONFERENCIA"]
escrever_cabecalho(ws, [
    "Conferência", "Valor calculado",
    "Valor esperado", "Situação"
])

controles = [
    ("Empenhado julho/2025", "=COMPARATIVO!C10", 183545888.90),
    ("Empenhado dezembro/2025", "=COMPARATIVO!D10", 280532572.59),
    ("Empenhado julho/2026", "=COMPARATIVO!I10", 196506138.54),
    ("Dotação atualizada julho/2026", "=COMPARATIVO!J10", 303992427.31),
    ("Proposta LOA 2027", "=COMPARATIVO!K10", 324215000.00),
    ("Ajuste líquido julho/2025", "=COMPARATIVO!E10", 0.00),
    ("Ajuste líquido dezembro/2025", "=COMPARATIVO!F10", 0.00),
    ("Administração + FAPEN jul/2026", "=FAPEN!E4", 44098013.49)
]

for i, (descricao, formula, esperado) in enumerate(
    controles, start=2
):
    ws.append([
        descricao, formula, esperado,
        f'=IF(ABS(B{i}-C{i})<0.01,"OK","DIVERGÊNCIA")'
    ])

finalizar_aba(
    ws, {"A": 47, "B": 26, "C": 26, "D": 20},
    ["B", "C"]
)

# 13. Ajustes finais de visualização
for ws in wb.worksheets:
    ws.sheet_view.showGridLines = False

    for row in ws:
        for cell in row:
            if cell.data_type == "f":
                cell.font = Font(color="17324D")

# Reforçar a identificação dos totais
ws = abas["COMPARATIVO"]
for cell in ws[10]:
    cell.fill = PatternFill("solid", fgColor=azul_claro)
    cell.font = Font(color=azul, bold=True)

# Atualizar fórmulas ao abrir no Excel
wb.calculation = CalcProperties(
    calcMode="auto",
    fullCalcOnLoad=True,
    forceFullCalc=True
)

# 14. Salvar SEM sobrescrever a planilha anterior
nome_arquivo = "Comparativo_Secretarias_Cerquilho_Auditavel.xlsx"

pasta_drive = (
    "/content/drive/MyDrive/"
    "Analise_Orcamentaria_Cerquilho"
)

if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(pasta_drive, exist_ok=True)
    caminho = os.path.join(pasta_drive, nome_arquivo)
else:
    caminho = os.path.join("/content", nome_arquivo)

wb.save(caminho)

print("PLANILHA GERADA COM SUCESSO")
print("Arquivo:", caminho)
print("Abas:", ", ".join(wb.sheetnames))
print("Secretarias:", 8)
print("Encargos originais registrados:", 15)
print("Fórmulas: incluídas no Excel")
print("Recalcular: ao abrir no Excel")

# Download
files.download(caminho)


PLANILHA GERADA COM SUCESSO
Arquivo: /content/Comparativo_Secretarias_Cerquilho_Auditavel.xlsx
Abas: LEIA_ME, HISTORICO_2025, EXECUCAO_2026, LOA_2027, ENCARGOS, COMPARATIVO, FAPEN, CONFERENCIA
Secretarias: 8
Encargos originais registrados: 15
Fórmulas: incluídas no Excel
Recalcular: ao abrir no Excel


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>